# *Batch size*

# Batch size 32 

In [8]:
# =================== IMPORTS ===================
import os
import numpy as np
import random
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, backend as K
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from keras.optimizers import AdamW
from tensorflow.keras.losses import CategoricalCrossentropy
from shutil import rmtree, copyfile
from tqdm import tqdm
from tensorflow.keras import mixed_precision

# =================== CUSTOM LAYERS ===================
@tf.keras.utils.register_keras_serializable()
class MeanChannel(layers.Layer):
    def call(self, inputs):
        return tf.reduce_mean(inputs, axis=-1, keepdims=True)

@tf.keras.utils.register_keras_serializable()
class MaxChannel(layers.Layer):
    def call(self, inputs):
        return tf.reduce_max(inputs, axis=-1, keepdims=True)

# =================== ENABLE MIXED PRECISION ===================
mixed_precision.set_global_policy('mixed_float16')

# =================== CONSTANTS ===================
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 4
EPOCHS = 60
train_ratio, val_ratio = 0.7, 0.15

# =================== DIRECTORIES ===================
dataset_dir = '/kaggle/input/colon-augmented-custom/colon_augmented_png_v9'
base_dir = '/kaggle/working/colon_split'

# =================== CLEAN EXISTING SPLITS ===================
if os.path.exists(base_dir):
    for folder in os.listdir(base_dir):
        rmtree(os.path.join(base_dir, folder))

# =================== DATA SPLIT ===================
splits = ['train', 'val', 'test']
class_names = os.listdir(dataset_dir)

for split in splits:
    for class_name in class_names:
        os.makedirs(os.path.join(base_dir, split, class_name), exist_ok=True)

for class_name in class_names:
    image_list = os.listdir(os.path.join(dataset_dir, class_name))
    random.shuffle(image_list)

    n_total = len(image_list)
    n_train = int(train_ratio * n_total)
    n_val = int(val_ratio * n_total)

    train_files = image_list[:n_train]
    val_files = image_list[n_train:n_train + n_val]
    test_files = image_list[n_train + n_val:]

    for split, split_files in zip(['train', 'val', 'test'], [train_files, val_files, test_files]):
        for img in tqdm(split_files, desc=f'{class_name} - {split}'):
            src = os.path.join(dataset_dir, class_name, img)
            dst = os.path.join(base_dir, split, class_name, img)
            copyfile(src, dst)

# =================== DATA GENERATORS ===================
train_datagen = ImageDataGenerator(
    rescale=1./255,
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input
)
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    os.path.join(base_dir, 'train'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_generator = val_test_datagen.flow_from_directory(
    os.path.join(base_dir, 'val'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_generator = val_test_datagen.flow_from_directory(
    os.path.join(base_dir, 'test'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# =================== MODEL BUILDING ===================
def cbam_block(input_feature, ratio=8):
    channel = input_feature.shape[-1]

    avg_pool = layers.GlobalAveragePooling2D()(input_feature)
    max_pool = layers.GlobalMaxPooling2D()(input_feature)

    shared_dense_one = layers.Dense(channel // ratio, activation='relu')
    shared_dense_two = layers.Dense(channel)

    mlp_avg = shared_dense_two(shared_dense_one(avg_pool))
    mlp_max = shared_dense_two(shared_dense_one(max_pool))

    channel_attention = layers.Add()([mlp_avg, mlp_max])
    channel_attention = layers.Activation('sigmoid')(channel_attention)
    channel_attention = layers.Reshape((1, 1, channel))(channel_attention)

    x = layers.Multiply()([input_feature, channel_attention])

    avg_pool_spatial = MeanChannel()(x)
    max_pool_spatial = MaxChannel()(x)
    concat = layers.Concatenate(axis=-1)([avg_pool_spatial, max_pool_spatial])
    spatial_attention = layers.Conv2D(1, kernel_size=7, padding='same', activation='sigmoid')(concat)

    refined_feature = layers.Multiply()([x, spatial_attention])
    return refined_feature

def build_model():
    base_model = tf.keras.applications.EfficientNetB0(
        include_top=False, input_shape=(224, 224, 3), weights='imagenet'
    )
    base_model.trainable = True

    inputs = layers.Input(shape=(224, 224, 3))
    x = base_model(inputs, training=True)
    x = cbam_block(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)

    model = tf.keras.Model(inputs, outputs)
    return model

model = build_model()

# =================== OPTIMIZER & LOSS ===================
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1.5e-3,
    decay_steps=1000,
    alpha=1e-6
)
optimizer = AdamW(learning_rate=lr_schedule, weight_decay=1e-5)
loss_fn = CategoricalCrossentropy(label_smoothing=0.1)

model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])

# =================== CALLBACKS ===================
checkpoint_cb = callbacks.ModelCheckpoint(
    'carenet.keras', monitor='val_accuracy', save_best_only=True, verbose=1
)
earlystop_cb = callbacks.EarlyStopping(
    monitor='val_accuracy', patience=7, restore_best_weights=True
)

# =================== TRAINING ===================
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=[checkpoint_cb, earlystop_cb],
    verbose=1
)

# =================== EVALUATION ===================
model.load_weights("carenet.keras")
test_loss, test_acc = model.evaluate(test_generator)
print(f"\n✅ Test Accuracy: {test_acc:.4f}")


1_ulcerative_colitis - test: 100%|██████████| 225/225 [00:00<00:00, 398.65it/s]


Found 4200 images belonging to 4 classes.
Found 900 images belonging to 4 classes.
Found 900 images belonging to 4 classes.
Epoch 1/60
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 434ms/step - accuracy: 0.6146 - loss: 1.0420
Epoch 1: val_accuracy improved from -inf to 0.25000, saving model to carenet.keras
132/132 ━━━━━━━━━━━━━━━━━━━━ 186s 560ms/step - accuracy: 0.6158 - loss: 1.0401 - val_accuracy: 0.2500 - val_loss: 1.4077
Epoch 2/60
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - accuracy: 0.9522 - loss: 0.4686
Epoch 2: val_accuracy improved from 0.25000 to 0.49333, saving model to carenet.keras
132/132 ━━━━━━━━━━━━━━━━━━━━ 15s 114ms/step - accuracy: 0.9522 - loss: 0.4686 - val_accuracy: 0.4933 - val_loss: 1.2334
Epoch 3/60
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - accuracy: 0.9839 - loss: 0.4079
Epoch 3: val_accuracy improved from 0.49333 to 0.68000, saving model to carenet.keras
132/132 ━━━━━━━━━━━━━━━━━━━━ 15s 112ms/step - accuracy: 0.9838 - loss: 0.4079 - val_accuracy: 0.6800 - val_loss: 1.022

# Batch size 16

In [9]:
# =================== IMPORTS ===================
import os
import numpy as np
import random
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, backend as K
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from keras.optimizers import AdamW
from tensorflow.keras.losses import CategoricalCrossentropy
from shutil import rmtree, copyfile
from tqdm import tqdm
from tensorflow.keras import mixed_precision

# =================== CUSTOM LAYERS ===================
@tf.keras.utils.register_keras_serializable()
class MeanChannel(layers.Layer):
    def call(self, inputs):
        return tf.reduce_mean(inputs, axis=-1, keepdims=True)

@tf.keras.utils.register_keras_serializable()
class MaxChannel(layers.Layer):
    def call(self, inputs):
        return tf.reduce_max(inputs, axis=-1, keepdims=True)

# =================== ENABLE MIXED PRECISION ===================
mixed_precision.set_global_policy('mixed_float16')

# =================== CONSTANTS ===================
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
NUM_CLASSES = 4
EPOCHS = 60
train_ratio, val_ratio = 0.7, 0.15

# =================== DIRECTORIES ===================
dataset_dir = '/kaggle/input/colon-augmented-custom/colon_augmented_png_v9'
base_dir = '/kaggle/working/colon_split'

# =================== CLEAN EXISTING SPLITS ===================
if os.path.exists(base_dir):
    for folder in os.listdir(base_dir):
        rmtree(os.path.join(base_dir, folder))

# =================== DATA SPLIT ===================
splits = ['train', 'val', 'test']
class_names = os.listdir(dataset_dir)

for split in splits:
    for class_name in class_names:
        os.makedirs(os.path.join(base_dir, split, class_name), exist_ok=True)

for class_name in class_names:
    image_list = os.listdir(os.path.join(dataset_dir, class_name))
    random.shuffle(image_list)

    n_total = len(image_list)
    n_train = int(train_ratio * n_total)
    n_val = int(val_ratio * n_total)

    train_files = image_list[:n_train]
    val_files = image_list[n_train:n_train + n_val]
    test_files = image_list[n_train + n_val:]

    for split, split_files in zip(['train', 'val', 'test'], [train_files, val_files, test_files]):
        for img in tqdm(split_files, desc=f'{class_name} - {split}'):
            src = os.path.join(dataset_dir, class_name, img)
            dst = os.path.join(base_dir, split, class_name, img)
            copyfile(src, dst)

# =================== DATA GENERATORS ===================
train_datagen = ImageDataGenerator(
    rescale=1./255,
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input
)
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    os.path.join(base_dir, 'train'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_generator = val_test_datagen.flow_from_directory(
    os.path.join(base_dir, 'val'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_generator = val_test_datagen.flow_from_directory(
    os.path.join(base_dir, 'test'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# =================== MODEL BUILDING ===================
def cbam_block(input_feature, ratio=8):
    channel = input_feature.shape[-1]

    avg_pool = layers.GlobalAveragePooling2D()(input_feature)
    max_pool = layers.GlobalMaxPooling2D()(input_feature)

    shared_dense_one = layers.Dense(channel // ratio, activation='relu')
    shared_dense_two = layers.Dense(channel)

    mlp_avg = shared_dense_two(shared_dense_one(avg_pool))
    mlp_max = shared_dense_two(shared_dense_one(max_pool))

    channel_attention = layers.Add()([mlp_avg, mlp_max])
    channel_attention = layers.Activation('sigmoid')(channel_attention)
    channel_attention = layers.Reshape((1, 1, channel))(channel_attention)

    x = layers.Multiply()([input_feature, channel_attention])

    avg_pool_spatial = MeanChannel()(x)
    max_pool_spatial = MaxChannel()(x)
    concat = layers.Concatenate(axis=-1)([avg_pool_spatial, max_pool_spatial])
    spatial_attention = layers.Conv2D(1, kernel_size=7, padding='same', activation='sigmoid')(concat)

    refined_feature = layers.Multiply()([x, spatial_attention])
    return refined_feature

def build_model():
    base_model = tf.keras.applications.EfficientNetB0(
        include_top=False, input_shape=(224, 224, 3), weights='imagenet'
    )
    base_model.trainable = True

    inputs = layers.Input(shape=(224, 224, 3))
    x = base_model(inputs, training=True)
    x = cbam_block(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)

    model = tf.keras.Model(inputs, outputs)
    return model

model = build_model()

# =================== OPTIMIZER & LOSS ===================
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1.5e-3,
    decay_steps=1000,
    alpha=1e-6
)
optimizer = AdamW(learning_rate=lr_schedule, weight_decay=1e-5)
loss_fn = CategoricalCrossentropy(label_smoothing=0.1)

model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])

# =================== CALLBACKS ===================
checkpoint_cb = callbacks.ModelCheckpoint(
    'carenet.keras', monitor='val_accuracy', save_best_only=True, verbose=1
)
earlystop_cb = callbacks.EarlyStopping(
    monitor='val_accuracy', patience=7, restore_best_weights=True
)

# =================== TRAINING ===================
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=[checkpoint_cb, earlystop_cb],
    verbose=1
)

# =================== EVALUATION ===================
model.load_weights("carenet.keras")
test_loss, test_acc = model.evaluate(test_generator)
print(f"\n✅ Test Accuracy: {test_acc:.4f}")


1_ulcerative_colitis - test: 100%|██████████| 225/225 [00:00<00:00, 449.22it/s]


Found 4200 images belonging to 4 classes.
Found 900 images belonging to 4 classes.
Found 900 images belonging to 4 classes.
Epoch 1/60
262/263 ━━━━━━━━━━━━━━━━━━━━ 0s 218ms/step - accuracy: 0.6390 - loss: 0.9941
Epoch 1: val_accuracy improved from -inf to 0.25889, saving model to carenet.keras
263/263 ━━━━━━━━━━━━━━━━━━━━ 190s 283ms/step - accuracy: 0.6402 - loss: 0.9922 - val_accuracy: 0.2589 - val_loss: 1.3828
Epoch 2/60
262/263 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.9437 - loss: 0.4804
Epoch 2: val_accuracy improved from 0.25889 to 0.87333, saving model to carenet.keras
263/263 ━━━━━━━━━━━━━━━━━━━━ 15s 58ms/step - accuracy: 0.9438 - loss: 0.4803 - val_accuracy: 0.8733 - val_loss: 0.6587
Epoch 3/60
263/263 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.9768 - loss: 0.4240
Epoch 3: val_accuracy did not improve from 0.87333
263/263 ━━━━━━━━━━━━━━━━━━━━ 14s 52ms/step - accuracy: 0.9768 - loss: 0.4240 - val_accuracy: 0.8733 - val_loss: 0.6385
Epoch 4/60
263/263 ━━━━━━━━━━━━━━━━

# Batch size 64

In [10]:
# =================== IMPORTS ===================
import os
import numpy as np
import random
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, backend as K
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from keras.optimizers import AdamW
from tensorflow.keras.losses import CategoricalCrossentropy
from shutil import rmtree, copyfile
from tqdm import tqdm
from tensorflow.keras import mixed_precision

# =================== CUSTOM LAYERS ===================
@tf.keras.utils.register_keras_serializable()
class MeanChannel(layers.Layer):
    def call(self, inputs):
        return tf.reduce_mean(inputs, axis=-1, keepdims=True)

@tf.keras.utils.register_keras_serializable()
class MaxChannel(layers.Layer):
    def call(self, inputs):
        return tf.reduce_max(inputs, axis=-1, keepdims=True)

# =================== ENABLE MIXED PRECISION ===================
mixed_precision.set_global_policy('mixed_float16')

# =================== CONSTANTS ===================
IMG_SIZE = (224, 224)
BATCH_SIZE = 64
NUM_CLASSES = 4
EPOCHS = 60
train_ratio, val_ratio = 0.7, 0.15

# =================== DIRECTORIES ===================
dataset_dir = '/kaggle/input/colon-augmented-custom/colon_augmented_png_v9'
base_dir = '/kaggle/working/colon_split'

# =================== CLEAN EXISTING SPLITS ===================
if os.path.exists(base_dir):
    for folder in os.listdir(base_dir):
        rmtree(os.path.join(base_dir, folder))

# =================== DATA SPLIT ===================
splits = ['train', 'val', 'test']
class_names = os.listdir(dataset_dir)

for split in splits:
    for class_name in class_names:
        os.makedirs(os.path.join(base_dir, split, class_name), exist_ok=True)

for class_name in class_names:
    image_list = os.listdir(os.path.join(dataset_dir, class_name))
    random.shuffle(image_list)

    n_total = len(image_list)
    n_train = int(train_ratio * n_total)
    n_val = int(val_ratio * n_total)

    train_files = image_list[:n_train]
    val_files = image_list[n_train:n_train + n_val]
    test_files = image_list[n_train + n_val:]

    for split, split_files in zip(['train', 'val', 'test'], [train_files, val_files, test_files]):
        for img in tqdm(split_files, desc=f'{class_name} - {split}'):
            src = os.path.join(dataset_dir, class_name, img)
            dst = os.path.join(base_dir, split, class_name, img)
            copyfile(src, dst)

# =================== DATA GENERATORS ===================
train_datagen = ImageDataGenerator(
    rescale=1./255,
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input
)
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    os.path.join(base_dir, 'train'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_generator = val_test_datagen.flow_from_directory(
    os.path.join(base_dir, 'val'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_generator = val_test_datagen.flow_from_directory(
    os.path.join(base_dir, 'test'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# =================== MODEL BUILDING ===================
def cbam_block(input_feature, ratio=8):
    channel = input_feature.shape[-1]

    avg_pool = layers.GlobalAveragePooling2D()(input_feature)
    max_pool = layers.GlobalMaxPooling2D()(input_feature)

    shared_dense_one = layers.Dense(channel // ratio, activation='relu')
    shared_dense_two = layers.Dense(channel)

    mlp_avg = shared_dense_two(shared_dense_one(avg_pool))
    mlp_max = shared_dense_two(shared_dense_one(max_pool))

    channel_attention = layers.Add()([mlp_avg, mlp_max])
    channel_attention = layers.Activation('sigmoid')(channel_attention)
    channel_attention = layers.Reshape((1, 1, channel))(channel_attention)

    x = layers.Multiply()([input_feature, channel_attention])

    avg_pool_spatial = MeanChannel()(x)
    max_pool_spatial = MaxChannel()(x)
    concat = layers.Concatenate(axis=-1)([avg_pool_spatial, max_pool_spatial])
    spatial_attention = layers.Conv2D(1, kernel_size=7, padding='same', activation='sigmoid')(concat)

    refined_feature = layers.Multiply()([x, spatial_attention])
    return refined_feature

def build_model():
    base_model = tf.keras.applications.EfficientNetB0(
        include_top=False, input_shape=(224, 224, 3), weights='imagenet'
    )
    base_model.trainable = True

    inputs = layers.Input(shape=(224, 224, 3))
    x = base_model(inputs, training=True)
    x = cbam_block(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)

    model = tf.keras.Model(inputs, outputs)
    return model

model = build_model()

# =================== OPTIMIZER & LOSS ===================
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1.5e-3,
    decay_steps=1000,
    alpha=1e-6
)
optimizer = AdamW(learning_rate=lr_schedule, weight_decay=1e-5)
loss_fn = CategoricalCrossentropy(label_smoothing=0.1)

model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])

# =================== CALLBACKS ===================
checkpoint_cb = callbacks.ModelCheckpoint(
    'carenet.keras', monitor='val_accuracy', save_best_only=True, verbose=1
)
earlystop_cb = callbacks.EarlyStopping(
    monitor='val_accuracy', patience=7, restore_best_weights=True
)

# =================== TRAINING ===================
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=[checkpoint_cb, earlystop_cb],
    verbose=1
)

# =================== EVALUATION ===================
model.load_weights("carenet.keras")
test_loss, test_acc = model.evaluate(test_generator)
print(f"\n✅ Test Accuracy: {test_acc:.4f}")


1_ulcerative_colitis - test: 100%|██████████| 225/225 [00:00<00:00, 435.00it/s]

Found 4200 images belonging to 4 classes.
Found 900 images belonging to 4 classes.
Found 900 images belonging to 4 classes.


Epoch 1/60
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 939ms/step - accuracy: 0.5042 - loss: 1.1863
Epoch 1: val_accuracy improved from -inf to 0.25000, saving model to carenet.keras
66/66 ━━━━━━━━━━━━━━━━━━━━ 194s 1s/step - accuracy: 0.5071 - loss: 1.1826 - val_accuracy: 0.2500 - val_loss: 1.4107
Epoch 2/60
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 193ms/step - accuracy: 0.9304 - loss: 0.5152
Epoch 2: val_accuracy did not improve from 0.25000
66/66 ━━━━━━━━━━━━━━━━━━━━ 16s 234ms/step - accuracy: 0.9306 - loss: 0.5149 - val_accuracy: 0.2489 - val_loss: 1.3794
Epoch 3/60
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 196ms/step - accuracy: 0.9761 - loss: 0.4238
Epoch 3: val_accuracy improved from 0.25000 to 0.37444, saving model to carenet.keras
66/66 ━━━━━━━━━━━━━━━━━━━━ 17s 260ms/step - accuracy: 0.9761 - loss: 0.4238 - val_accuracy: 0.3744 - val_loss: 1.3378
Epoch 4/60
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 210ms/step - accuracy: 0.9834 - loss: 0.4037
Epoch 4: val_accuracy improved from 0.37444 to 0.41444, saving model to carenet.kera

# Batch size 128

In [11]:
# =================== IMPORTS ===================
import os
import numpy as np
import random
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, backend as K
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from keras.optimizers import AdamW
from tensorflow.keras.losses import CategoricalCrossentropy
from shutil import rmtree, copyfile
from tqdm import tqdm
from tensorflow.keras import mixed_precision

# =================== CUSTOM LAYERS ===================
@tf.keras.utils.register_keras_serializable()
class MeanChannel(layers.Layer):
    def call(self, inputs):
        return tf.reduce_mean(inputs, axis=-1, keepdims=True)

@tf.keras.utils.register_keras_serializable()
class MaxChannel(layers.Layer):
    def call(self, inputs):
        return tf.reduce_max(inputs, axis=-1, keepdims=True)

# =================== ENABLE MIXED PRECISION ===================
mixed_precision.set_global_policy('mixed_float16')

# =================== CONSTANTS ===================
IMG_SIZE = (224, 224)
BATCH_SIZE = 128
NUM_CLASSES = 4
EPOCHS = 60
train_ratio, val_ratio = 0.7, 0.15

# =================== DIRECTORIES ===================
dataset_dir = '/kaggle/input/colon-augmented-custom/colon_augmented_png_v9'
base_dir = '/kaggle/working/colon_split'

# =================== CLEAN EXISTING SPLITS ===================
if os.path.exists(base_dir):
    for folder in os.listdir(base_dir):
        rmtree(os.path.join(base_dir, folder))

# =================== DATA SPLIT ===================
splits = ['train', 'val', 'test']
class_names = os.listdir(dataset_dir)

for split in splits:
    for class_name in class_names:
        os.makedirs(os.path.join(base_dir, split, class_name), exist_ok=True)

for class_name in class_names:
    image_list = os.listdir(os.path.join(dataset_dir, class_name))
    random.shuffle(image_list)

    n_total = len(image_list)
    n_train = int(train_ratio * n_total)
    n_val = int(val_ratio * n_total)

    train_files = image_list[:n_train]
    val_files = image_list[n_train:n_train + n_val]
    test_files = image_list[n_train + n_val:]

    for split, split_files in zip(['train', 'val', 'test'], [train_files, val_files, test_files]):
        for img in tqdm(split_files, desc=f'{class_name} - {split}'):
            src = os.path.join(dataset_dir, class_name, img)
            dst = os.path.join(base_dir, split, class_name, img)
            copyfile(src, dst)

# =================== DATA GENERATORS ===================
train_datagen = ImageDataGenerator(
    rescale=1./255,
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input
)
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    os.path.join(base_dir, 'train'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_generator = val_test_datagen.flow_from_directory(
    os.path.join(base_dir, 'val'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_generator = val_test_datagen.flow_from_directory(
    os.path.join(base_dir, 'test'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# =================== MODEL BUILDING ===================
def cbam_block(input_feature, ratio=8):
    channel = input_feature.shape[-1]

    avg_pool = layers.GlobalAveragePooling2D()(input_feature)
    max_pool = layers.GlobalMaxPooling2D()(input_feature)

    shared_dense_one = layers.Dense(channel // ratio, activation='relu')
    shared_dense_two = layers.Dense(channel)

    mlp_avg = shared_dense_two(shared_dense_one(avg_pool))
    mlp_max = shared_dense_two(shared_dense_one(max_pool))

    channel_attention = layers.Add()([mlp_avg, mlp_max])
    channel_attention = layers.Activation('sigmoid')(channel_attention)
    channel_attention = layers.Reshape((1, 1, channel))(channel_attention)

    x = layers.Multiply()([input_feature, channel_attention])

    avg_pool_spatial = MeanChannel()(x)
    max_pool_spatial = MaxChannel()(x)
    concat = layers.Concatenate(axis=-1)([avg_pool_spatial, max_pool_spatial])
    spatial_attention = layers.Conv2D(1, kernel_size=7, padding='same', activation='sigmoid')(concat)

    refined_feature = layers.Multiply()([x, spatial_attention])
    return refined_feature

def build_model():
    base_model = tf.keras.applications.EfficientNetB0(
        include_top=False, input_shape=(224, 224, 3), weights='imagenet'
    )
    base_model.trainable = True

    inputs = layers.Input(shape=(224, 224, 3))
    x = base_model(inputs, training=True)
    x = cbam_block(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)

    model = tf.keras.Model(inputs, outputs)
    return model

model = build_model()

# =================== OPTIMIZER & LOSS ===================
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1.5e-3,
    decay_steps=1000,
    alpha=1e-6
)
optimizer = AdamW(learning_rate=lr_schedule, weight_decay=1e-5)
loss_fn = CategoricalCrossentropy(label_smoothing=0.1)

model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])

# =================== CALLBACKS ===================
checkpoint_cb = callbacks.ModelCheckpoint(
    'carenet.keras', monitor='val_accuracy', save_best_only=True, verbose=1
)
earlystop_cb = callbacks.EarlyStopping(
    monitor='val_accuracy', patience=7, restore_best_weights=True
)

# =================== TRAINING ===================
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=[checkpoint_cb, earlystop_cb],
    verbose=1
)

# =================== EVALUATION ===================
model.load_weights("carenet.keras")
test_loss, test_acc = model.evaluate(test_generator)
print(f"\n✅ Test Accuracy: {test_acc:.4f}")


1_ulcerative_colitis - test: 100%|██████████| 225/225 [00:00<00:00, 408.07it/s]


Found 4200 images belonging to 4 classes.
Found 900 images belonging to 4 classes.
Found 900 images belonging to 4 classes.
Epoch 1/60
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.4247 - loss: 1.2805
Epoch 1: val_accuracy improved from -inf to 0.25000, saving model to carenet.keras
33/33 ━━━━━━━━━━━━━━━━━━━━ 203s 2s/step - accuracy: 0.4301 - loss: 1.2751 - val_accuracy: 0.2500 - val_loss: 1.3917
Epoch 2/60
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 432ms/step - accuracy: 0.8835 - loss: 0.6088
Epoch 2: val_accuracy did not improve from 0.25000
33/33 ━━━━━━━━━━━━━━━━━━━━ 18s 527ms/step - accuracy: 0.8840 - loss: 0.6075 - val_accuracy: 0.2422 - val_loss: 1.3896
Epoch 3/60
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 432ms/step - accuracy: 0.9581 - loss: 0.4567
Epoch 3: val_accuracy did not improve from 0.25000
33/33 ━━━━━━━━━━━━━━━━━━━━ 18s 526ms/step - accuracy: 0.9582 - loss: 0.4564 - val_accuracy: 0.2500 - val_loss: 1.3868
Epoch 4/60
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 433ms/step - accuracy: 0.9785 - loss: 0.420

# *Epochs*

# Epoch 40

In [1]:
# =================== IMPORTS ===================
import os
import numpy as np
import random
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, backend as K
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from keras.optimizers import AdamW
from tensorflow.keras.losses import CategoricalCrossentropy
from shutil import rmtree, copyfile
from tqdm import tqdm
from tensorflow.keras import mixed_precision

# =================== CUSTOM LAYERS ===================
@tf.keras.utils.register_keras_serializable()
class MeanChannel(layers.Layer):
    def call(self, inputs):
        return tf.reduce_mean(inputs, axis=-1, keepdims=True)

@tf.keras.utils.register_keras_serializable()
class MaxChannel(layers.Layer):
    def call(self, inputs):
        return tf.reduce_max(inputs, axis=-1, keepdims=True)

# =================== ENABLE MIXED PRECISION ===================
mixed_precision.set_global_policy('mixed_float16')

# =================== CONSTANTS ===================
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 4
EPOCHS = 40
train_ratio, val_ratio = 0.7, 0.15

# =================== DIRECTORIES ===================
dataset_dir = '/kaggle/input/colon-augmented-custom/colon_augmented_png_v9'
base_dir = '/kaggle/working/colon_split'

# =================== CLEAN EXISTING SPLITS ===================
if os.path.exists(base_dir):
    for folder in os.listdir(base_dir):
        rmtree(os.path.join(base_dir, folder))

# =================== DATA SPLIT ===================
splits = ['train', 'val', 'test']
class_names = os.listdir(dataset_dir)

for split in splits:
    for class_name in class_names:
        os.makedirs(os.path.join(base_dir, split, class_name), exist_ok=True)

for class_name in class_names:
    image_list = os.listdir(os.path.join(dataset_dir, class_name))
    random.shuffle(image_list)

    n_total = len(image_list)
    n_train = int(train_ratio * n_total)
    n_val = int(val_ratio * n_total)

    train_files = image_list[:n_train]
    val_files = image_list[n_train:n_train + n_val]
    test_files = image_list[n_train + n_val:]

    for split, split_files in zip(['train', 'val', 'test'], [train_files, val_files, test_files]):
        for img in tqdm(split_files, desc=f'{class_name} - {split}'):
            src = os.path.join(dataset_dir, class_name, img)
            dst = os.path.join(base_dir, split, class_name, img)
            copyfile(src, dst)

# =================== DATA GENERATORS ===================
train_datagen = ImageDataGenerator(
    rescale=1./255,
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input
)
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    os.path.join(base_dir, 'train'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_generator = val_test_datagen.flow_from_directory(
    os.path.join(base_dir, 'val'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_generator = val_test_datagen.flow_from_directory(
    os.path.join(base_dir, 'test'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# =================== MODEL BUILDING ===================
def cbam_block(input_feature, ratio=8):
    channel = input_feature.shape[-1]

    avg_pool = layers.GlobalAveragePooling2D()(input_feature)
    max_pool = layers.GlobalMaxPooling2D()(input_feature)

    shared_dense_one = layers.Dense(channel // ratio, activation='relu')
    shared_dense_two = layers.Dense(channel)

    mlp_avg = shared_dense_two(shared_dense_one(avg_pool))
    mlp_max = shared_dense_two(shared_dense_one(max_pool))

    channel_attention = layers.Add()([mlp_avg, mlp_max])
    channel_attention = layers.Activation('sigmoid')(channel_attention)
    channel_attention = layers.Reshape((1, 1, channel))(channel_attention)

    x = layers.Multiply()([input_feature, channel_attention])

    avg_pool_spatial = MeanChannel()(x)
    max_pool_spatial = MaxChannel()(x)
    concat = layers.Concatenate(axis=-1)([avg_pool_spatial, max_pool_spatial])
    spatial_attention = layers.Conv2D(1, kernel_size=7, padding='same', activation='sigmoid')(concat)

    refined_feature = layers.Multiply()([x, spatial_attention])
    return refined_feature

def build_model():
    base_model = tf.keras.applications.EfficientNetB0(
        include_top=False, input_shape=(224, 224, 3), weights='imagenet'
    )
    base_model.trainable = True

    inputs = layers.Input(shape=(224, 224, 3))
    x = base_model(inputs, training=True)
    x = cbam_block(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)

    model = tf.keras.Model(inputs, outputs)
    return model

model = build_model()

# =================== OPTIMIZER & LOSS ===================
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1.5e-3,
    decay_steps=1000,
    alpha=1e-6
)
optimizer = AdamW(learning_rate=lr_schedule, weight_decay=1e-5)
loss_fn = CategoricalCrossentropy(label_smoothing=0.1)

model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])

# =================== CALLBACKS ===================
checkpoint_cb = callbacks.ModelCheckpoint(
    'carenet.keras', monitor='val_accuracy', save_best_only=True, verbose=1
)
earlystop_cb = callbacks.EarlyStopping(
    monitor='val_accuracy', patience=7, restore_best_weights=True
)

# =================== TRAINING ===================
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=[checkpoint_cb, earlystop_cb],
    verbose=1
)

# =================== EVALUATION ===================
model.load_weights("carenet.keras")
test_loss, test_acc = model.evaluate(test_generator)
print(f"\n✅ Test Accuracy: {test_acc:.4f}")


2025-07-24 15:13:50.086169: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753370030.446673      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753370030.546619      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
1_ulcerative_colitis - test: 100%|██████████| 225/225 [00:02<00:00, 98.47it/s] 

Found 4200 images belonging to 4 classes.


Found 900 images belonging to 4 classes.
Found 900 images belonging to 4 classes.


I0000 00:00:1753370112.549181      36 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1753370112.549954      36 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/40


I0000 00:00:1753370182.638628     119 service.cc:148] XLA service 0x7c71b4002130 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1753370182.640315     119 service.cc:156]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1753370182.640338     119 service.cc:156]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1753370190.947058     119 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1753370242.503550     119 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 484ms/step - accuracy: 0.6411 - loss: 0.9927
Epoch 1: val_accuracy improved from -inf to 0.24778, saving model to carenet.keras
132/132 ━━━━━━━━━━━━━━━━━━━━ 209s 632ms/step - accuracy: 0.6423 - loss: 0.9909 - val_accuracy: 0.2478 - val_loss: 1.3953
Epoch 2/40
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.9479 - loss: 0.4872
Epoch 2: val_accuracy improved from 0.24778 to 0.44000, saving model to carenet.keras
132/132 ━━━━━━━━━━━━━━━━━━━━ 14s 109ms/step - accuracy: 0.9479 - loss: 0.4871 - val_accuracy: 0.4400 - val_loss: 1.3014
Epoch 3/40
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step - accuracy: 0.9708 - loss: 0.4368
Epoch 3: val_accuracy improved from 0.44000 to 0.57778, saving model to carenet.keras
132/132 ━━━━━━━━━━━━━━━━━━━━ 14s 109ms/step - accuracy: 0.9708 - loss: 0.4367 - val_accuracy: 0.5778 - val_loss: 1.1939
Epoch 4/40
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - accuracy: 0.9840 - loss: 0.4021
Epoch 4: val_accuracy improved from 0.57778 to

# Epoch 50

In [6]:
# =================== IMPORTS ===================
import os
import numpy as np
import random
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, backend as K
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from keras.optimizers import AdamW
from tensorflow.keras.losses import CategoricalCrossentropy
from shutil import rmtree, copyfile
from tqdm import tqdm
from tensorflow.keras import mixed_precision

# =================== CUSTOM LAYERS ===================
@tf.keras.utils.register_keras_serializable()
class MeanChannel(layers.Layer):
    def call(self, inputs):
        return tf.reduce_mean(inputs, axis=-1, keepdims=True)

@tf.keras.utils.register_keras_serializable()
class MaxChannel(layers.Layer):
    def call(self, inputs):
        return tf.reduce_max(inputs, axis=-1, keepdims=True)

# =================== ENABLE MIXED PRECISION ===================
mixed_precision.set_global_policy('mixed_float16')

# =================== CONSTANTS ===================
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 4
EPOCHS = 50
train_ratio, val_ratio = 0.7, 0.15

# =================== DIRECTORIES ===================
dataset_dir = '/kaggle/input/colon-augmented-custom/colon_augmented_png_v9'
base_dir = '/kaggle/working/colon_split'

# =================== CLEAN EXISTING SPLITS ===================
if os.path.exists(base_dir):
    for folder in os.listdir(base_dir):
        rmtree(os.path.join(base_dir, folder))

# =================== DATA SPLIT ===================
splits = ['train', 'val', 'test']
class_names = os.listdir(dataset_dir)

for split in splits:
    for class_name in class_names:
        os.makedirs(os.path.join(base_dir, split, class_name), exist_ok=True)

for class_name in class_names:
    image_list = os.listdir(os.path.join(dataset_dir, class_name))
    random.shuffle(image_list)

    n_total = len(image_list)
    n_train = int(train_ratio * n_total)
    n_val = int(val_ratio * n_total)

    train_files = image_list[:n_train]
    val_files = image_list[n_train:n_train + n_val]
    test_files = image_list[n_train + n_val:]

    for split, split_files in zip(['train', 'val', 'test'], [train_files, val_files, test_files]):
        for img in tqdm(split_files, desc=f'{class_name} - {split}'):
            src = os.path.join(dataset_dir, class_name, img)
            dst = os.path.join(base_dir, split, class_name, img)
            copyfile(src, dst)

# =================== DATA GENERATORS ===================
train_datagen = ImageDataGenerator(
    rescale=1./255,
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input
)
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    os.path.join(base_dir, 'train'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_generator = val_test_datagen.flow_from_directory(
    os.path.join(base_dir, 'val'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_generator = val_test_datagen.flow_from_directory(
    os.path.join(base_dir, 'test'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# =================== MODEL BUILDING ===================
def cbam_block(input_feature, ratio=8):
    channel = input_feature.shape[-1]

    avg_pool = layers.GlobalAveragePooling2D()(input_feature)
    max_pool = layers.GlobalMaxPooling2D()(input_feature)

    shared_dense_one = layers.Dense(channel // ratio, activation='relu')
    shared_dense_two = layers.Dense(channel)

    mlp_avg = shared_dense_two(shared_dense_one(avg_pool))
    mlp_max = shared_dense_two(shared_dense_one(max_pool))

    channel_attention = layers.Add()([mlp_avg, mlp_max])
    channel_attention = layers.Activation('sigmoid')(channel_attention)
    channel_attention = layers.Reshape((1, 1, channel))(channel_attention)

    x = layers.Multiply()([input_feature, channel_attention])

    avg_pool_spatial = MeanChannel()(x)
    max_pool_spatial = MaxChannel()(x)
    concat = layers.Concatenate(axis=-1)([avg_pool_spatial, max_pool_spatial])
    spatial_attention = layers.Conv2D(1, kernel_size=7, padding='same', activation='sigmoid')(concat)

    refined_feature = layers.Multiply()([x, spatial_attention])
    return refined_feature

def build_model():
    base_model = tf.keras.applications.EfficientNetB0(
        include_top=False, input_shape=(224, 224, 3), weights='imagenet'
    )
    base_model.trainable = True

    inputs = layers.Input(shape=(224, 224, 3))
    x = base_model(inputs, training=True)
    x = cbam_block(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)

    model = tf.keras.Model(inputs, outputs)
    return model

model = build_model()

# =================== OPTIMIZER & LOSS ===================
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1.5e-3,
    decay_steps=1000,
    alpha=1e-6
)
optimizer = AdamW(learning_rate=lr_schedule, weight_decay=1e-5)
loss_fn = CategoricalCrossentropy(label_smoothing=0.1)

model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])

# =================== CALLBACKS ===================
checkpoint_cb = callbacks.ModelCheckpoint(
    'carenet.keras', monitor='val_accuracy', save_best_only=True, verbose=1
)
earlystop_cb = callbacks.EarlyStopping(
    monitor='val_accuracy', patience=7, restore_best_weights=True
)

# =================== TRAINING ===================
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=[checkpoint_cb, earlystop_cb],
    verbose=1
)

# =================== EVALUATION ===================
model.load_weights("carenet.keras")
test_loss, test_acc = model.evaluate(test_generator)
print(f"\n✅ Test Accuracy: {test_acc:.4f}")


1_ulcerative_colitis - test: 100%|██████████| 225/225 [00:00<00:00, 404.66it/s]

Found 4200 images belonging to 4 classes.


Found 900 images belonging to 4 classes.
Found 900 images belonging to 4 classes.
Epoch 1/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 431ms/step - accuracy: 0.6405 - loss: 1.0353
Epoch 1: val_accuracy improved from -inf to 0.25000, saving model to carenet.keras
132/132 ━━━━━━━━━━━━━━━━━━━━ 190s 583ms/step - accuracy: 0.6417 - loss: 1.0332 - val_accuracy: 0.2500 - val_loss: 1.3924
Epoch 2/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step - accuracy: 0.9582 - loss: 0.4621
Epoch 2: val_accuracy improved from 0.25000 to 0.39667, saving model to carenet.keras
132/132 ━━━━━━━━━━━━━━━━━━━━ 14s 108ms/step - accuracy: 0.9582 - loss: 0.4620 - val_accuracy: 0.3967 - val_loss: 1.3456
Epoch 3/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - accuracy: 0.9770 - loss: 0.4111
Epoch 3: val_accuracy improved from 0.39667 to 0.58222, saving model to carenet.keras
132/132 ━━━━━━━━━━━━━━━━━━━━ 15s 115ms/step - accuracy: 0.9770 - loss: 0.4111 - val_accuracy: 0.5822 - val_loss: 1.1338
Epoch 4/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 

# Epoch 60

In [5]:
# =================== IMPORTS ===================
import os
import numpy as np
import random
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, backend as K
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from keras.optimizers import AdamW
from tensorflow.keras.losses import CategoricalCrossentropy
from shutil import rmtree, copyfile
from tqdm import tqdm
from tensorflow.keras import mixed_precision

# =================== CUSTOM LAYERS ===================
@tf.keras.utils.register_keras_serializable()
class MeanChannel(layers.Layer):
    def call(self, inputs):
        return tf.reduce_mean(inputs, axis=-1, keepdims=True)

@tf.keras.utils.register_keras_serializable()
class MaxChannel(layers.Layer):
    def call(self, inputs):
        return tf.reduce_max(inputs, axis=-1, keepdims=True)

# =================== ENABLE MIXED PRECISION ===================
mixed_precision.set_global_policy('mixed_float16')

# =================== CONSTANTS ===================
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 4
EPOCHS = 60
train_ratio, val_ratio = 0.7, 0.15

# =================== DIRECTORIES ===================
dataset_dir = '/kaggle/input/colon-augmented-custom/colon_augmented_png_v9'
base_dir = '/kaggle/working/colon_split'

# =================== CLEAN EXISTING SPLITS ===================
if os.path.exists(base_dir):
    for folder in os.listdir(base_dir):
        rmtree(os.path.join(base_dir, folder))

# =================== DATA SPLIT ===================
splits = ['train', 'val', 'test']
class_names = os.listdir(dataset_dir)

for split in splits:
    for class_name in class_names:
        os.makedirs(os.path.join(base_dir, split, class_name), exist_ok=True)

for class_name in class_names:
    image_list = os.listdir(os.path.join(dataset_dir, class_name))
    random.shuffle(image_list)

    n_total = len(image_list)
    n_train = int(train_ratio * n_total)
    n_val = int(val_ratio * n_total)

    train_files = image_list[:n_train]
    val_files = image_list[n_train:n_train + n_val]
    test_files = image_list[n_train + n_val:]

    for split, split_files in zip(['train', 'val', 'test'], [train_files, val_files, test_files]):
        for img in tqdm(split_files, desc=f'{class_name} - {split}'):
            src = os.path.join(dataset_dir, class_name, img)
            dst = os.path.join(base_dir, split, class_name, img)
            copyfile(src, dst)

# =================== DATA GENERATORS ===================
train_datagen = ImageDataGenerator(
    rescale=1./255,
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input
)
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    os.path.join(base_dir, 'train'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_generator = val_test_datagen.flow_from_directory(
    os.path.join(base_dir, 'val'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_generator = val_test_datagen.flow_from_directory(
    os.path.join(base_dir, 'test'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# =================== MODEL BUILDING ===================
def cbam_block(input_feature, ratio=8):
    channel = input_feature.shape[-1]

    avg_pool = layers.GlobalAveragePooling2D()(input_feature)
    max_pool = layers.GlobalMaxPooling2D()(input_feature)

    shared_dense_one = layers.Dense(channel // ratio, activation='relu')
    shared_dense_two = layers.Dense(channel)

    mlp_avg = shared_dense_two(shared_dense_one(avg_pool))
    mlp_max = shared_dense_two(shared_dense_one(max_pool))

    channel_attention = layers.Add()([mlp_avg, mlp_max])
    channel_attention = layers.Activation('sigmoid')(channel_attention)
    channel_attention = layers.Reshape((1, 1, channel))(channel_attention)

    x = layers.Multiply()([input_feature, channel_attention])

    avg_pool_spatial = MeanChannel()(x)
    max_pool_spatial = MaxChannel()(x)
    concat = layers.Concatenate(axis=-1)([avg_pool_spatial, max_pool_spatial])
    spatial_attention = layers.Conv2D(1, kernel_size=7, padding='same', activation='sigmoid')(concat)

    refined_feature = layers.Multiply()([x, spatial_attention])
    return refined_feature

def build_model():
    base_model = tf.keras.applications.EfficientNetB0(
        include_top=False, input_shape=(224, 224, 3), weights='imagenet'
    )
    base_model.trainable = True

    inputs = layers.Input(shape=(224, 224, 3))
    x = base_model(inputs, training=True)
    x = cbam_block(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)

    model = tf.keras.Model(inputs, outputs)
    return model

model = build_model()

# =================== OPTIMIZER & LOSS ===================
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1.5e-3,
    decay_steps=1000,
    alpha=1e-6
)
optimizer = AdamW(learning_rate=lr_schedule, weight_decay=1e-5)
loss_fn = CategoricalCrossentropy(label_smoothing=0.1)

model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])

# =================== CALLBACKS ===================
checkpoint_cb = callbacks.ModelCheckpoint(
    'carenet.keras', monitor='val_accuracy', save_best_only=True, verbose=1
)
earlystop_cb = callbacks.EarlyStopping(
    monitor='val_accuracy', patience=7, restore_best_weights=True
)

# =================== TRAINING ===================
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=[checkpoint_cb, earlystop_cb],
    verbose=1
)

# =================== EVALUATION ===================
model.load_weights("carenet.keras")
test_loss, test_acc = model.evaluate(test_generator)
print(f"\n✅ Test Accuracy: {test_acc:.4f}")


1_ulcerative_colitis - test: 100%|██████████| 225/225 [00:00<00:00, 303.12it/s]


Found 4200 images belonging to 4 classes.
Found 900 images belonging to 4 classes.
Found 900 images belonging to 4 classes.
Epoch 1/60
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 428ms/step - accuracy: 0.6364 - loss: 0.9862
Epoch 1: val_accuracy improved from -inf to 0.26333, saving model to carenet.keras
132/132 ━━━━━━━━━━━━━━━━━━━━ 187s 560ms/step - accuracy: 0.6376 - loss: 0.9844 - val_accuracy: 0.2633 - val_loss: 1.3913
Epoch 2/60
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - accuracy: 0.9450 - loss: 0.4898
Epoch 2: val_accuracy improved from 0.26333 to 0.36333, saving model to carenet.keras
132/132 ━━━━━━━━━━━━━━━━━━━━ 15s 112ms/step - accuracy: 0.9450 - loss: 0.4897 - val_accuracy: 0.3633 - val_loss: 1.3261
Epoch 3/60
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step - accuracy: 0.9742 - loss: 0.4300
Epoch 3: val_accuracy improved from 0.36333 to 0.55778, saving model to carenet.keras
132/132 ━━━━━━━━━━━━━━━━━━━━ 15s 109ms/step - accuracy: 0.9742 - loss: 0.4300 - val_accuracy: 0.5578 - val_loss: 1.100

# *Classifier Head*

# Classifier Head - GLOBAL AVERAGE POOLING2D

In [7]:
# =================== IMPORTS ===================
import os
import numpy as np
import random
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, backend as K
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from keras.optimizers import AdamW
from tensorflow.keras.losses import CategoricalCrossentropy
from shutil import rmtree, copyfile
from tqdm import tqdm
from tensorflow.keras import mixed_precision

# =================== CUSTOM LAYERS ===================
@tf.keras.utils.register_keras_serializable()
class MeanChannel(layers.Layer):
    def call(self, inputs):
        return tf.reduce_mean(inputs, axis=-1, keepdims=True)

@tf.keras.utils.register_keras_serializable()
class MaxChannel(layers.Layer):
    def call(self, inputs):
        return tf.reduce_max(inputs, axis=-1, keepdims=True)

# =================== ENABLE MIXED PRECISION ===================
mixed_precision.set_global_policy('mixed_float16')

# =================== CONSTANTS ===================
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 4
EPOCHS = 60
train_ratio, val_ratio = 0.7, 0.15

# =================== DIRECTORIES ===================
dataset_dir = '/kaggle/input/colon-augmented-custom/colon_augmented_png_v9'
base_dir = '/kaggle/working/colon_split'

# =================== CLEAN EXISTING SPLITS ===================
if os.path.exists(base_dir):
    for folder in os.listdir(base_dir):
        rmtree(os.path.join(base_dir, folder))

# =================== DATA SPLIT ===================
splits = ['train', 'val', 'test']
class_names = os.listdir(dataset_dir)

for split in splits:
    for class_name in class_names:
        os.makedirs(os.path.join(base_dir, split, class_name), exist_ok=True)

for class_name in class_names:
    image_list = os.listdir(os.path.join(dataset_dir, class_name))
    random.shuffle(image_list)

    n_total = len(image_list)
    n_train = int(train_ratio * n_total)
    n_val = int(val_ratio * n_total)

    train_files = image_list[:n_train]
    val_files = image_list[n_train:n_train + n_val]
    test_files = image_list[n_train + n_val:]

    for split, split_files in zip(['train', 'val', 'test'], [train_files, val_files, test_files]):
        for img in tqdm(split_files, desc=f'{class_name} - {split}'):
            src = os.path.join(dataset_dir, class_name, img)
            dst = os.path.join(base_dir, split, class_name, img)
            copyfile(src, dst)

# =================== DATA GENERATORS ===================
train_datagen = ImageDataGenerator(
    rescale=1./255,
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input
)
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    os.path.join(base_dir, 'train'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_generator = val_test_datagen.flow_from_directory(
    os.path.join(base_dir, 'val'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_generator = val_test_datagen.flow_from_directory(
    os.path.join(base_dir, 'test'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# =================== MODEL BUILDING ===================
def cbam_block(input_feature, ratio=8):
    channel = input_feature.shape[-1]

    avg_pool = layers.GlobalAveragePooling2D()(input_feature)
    max_pool = layers.GlobalMaxPooling2D()(input_feature)

    shared_dense_one = layers.Dense(channel // ratio, activation='relu')
    shared_dense_two = layers.Dense(channel)

    mlp_avg = shared_dense_two(shared_dense_one(avg_pool))
    mlp_max = shared_dense_two(shared_dense_one(max_pool))

    channel_attention = layers.Add()([mlp_avg, mlp_max])
    channel_attention = layers.Activation('sigmoid')(channel_attention)
    channel_attention = layers.Reshape((1, 1, channel))(channel_attention)

    x = layers.Multiply()([input_feature, channel_attention])

    avg_pool_spatial = MeanChannel()(x)
    max_pool_spatial = MaxChannel()(x)
    concat = layers.Concatenate(axis=-1)([avg_pool_spatial, max_pool_spatial])
    spatial_attention = layers.Conv2D(1, kernel_size=7, padding='same', activation='sigmoid')(concat)

    refined_feature = layers.Multiply()([x, spatial_attention])
    return refined_feature

def build_model():
    base_model = tf.keras.applications.EfficientNetB0(
        include_top=False, input_shape=(224, 224, 3), weights='imagenet'
    )
    base_model.trainable = True

    inputs = layers.Input(shape=(224, 224, 3))
    x = base_model(inputs, training=True)
    x = cbam_block(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)

    model = tf.keras.Model(inputs, outputs)
    return model

model = build_model()

# =================== OPTIMIZER & LOSS ===================
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1.5e-3,
    decay_steps=1000,
    alpha=1e-6
)
optimizer = AdamW(learning_rate=lr_schedule, weight_decay=1e-5)
loss_fn = CategoricalCrossentropy(label_smoothing=0.1)

model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])

# =================== CALLBACKS ===================
checkpoint_cb = callbacks.ModelCheckpoint(
    'carenet.keras', monitor='val_accuracy', save_best_only=True, verbose=1
)
earlystop_cb = callbacks.EarlyStopping(
    monitor='val_accuracy', patience=7, restore_best_weights=True
)

# =================== TRAINING ===================
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=[checkpoint_cb, earlystop_cb],
    verbose=1
)

# =================== EVALUATION ===================
model.load_weights("carenet.keras")
test_loss, test_acc = model.evaluate(test_generator)
print(f"\n✅ Test Accuracy: {test_acc:.4f}")


1_ulcerative_colitis - test: 100%|██████████| 225/225 [00:00<00:00, 425.94it/s]

Found 4200 images belonging to 4 classes.
Found 900 images belonging to 4 classes.


Found 900 images belonging to 4 classes.
Epoch 1/60
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 440ms/step - accuracy: 0.6563 - loss: 1.0044
Epoch 1: val_accuracy improved from -inf to 0.25778, saving model to carenet.keras
132/132 ━━━━━━━━━━━━━━━━━━━━ 191s 586ms/step - accuracy: 0.6574 - loss: 1.0026 - val_accuracy: 0.2578 - val_loss: 1.4156
Epoch 2/60
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - accuracy: 0.9584 - loss: 0.4668
Epoch 2: val_accuracy improved from 0.25778 to 0.49333, saving model to carenet.keras
132/132 ━━━━━━━━━━━━━━━━━━━━ 15s 114ms/step - accuracy: 0.9584 - loss: 0.4667 - val_accuracy: 0.4933 - val_loss: 1.2441
Epoch 3/60
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - accuracy: 0.9762 - loss: 0.4258
Epoch 3: val_accuracy improved from 0.49333 to 0.65333, saving model to carenet.keras
132/132 ━━━━━━━━━━━━━━━━━━━━ 15s 110ms/step - accuracy: 0.9762 - loss: 0.4257 - val_accuracy: 0.6533 - val_loss: 1.0760
Epoch 4/60
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - accuracy: 0.9936 - loss: 0

# Classifier Head - GlobalMaxPooling2D

In [8]:
# =================== IMPORTS ===================
import os
import numpy as np
import random
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, backend as K
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from keras.optimizers import AdamW
from tensorflow.keras.losses import CategoricalCrossentropy
from shutil import rmtree, copyfile
from tqdm import tqdm
from tensorflow.keras import mixed_precision

# =================== CUSTOM LAYERS ===================
@tf.keras.utils.register_keras_serializable()
class MeanChannel(layers.Layer):
    def call(self, inputs):
        return tf.reduce_mean(inputs, axis=-1, keepdims=True)

@tf.keras.utils.register_keras_serializable()
class MaxChannel(layers.Layer):
    def call(self, inputs):
        return tf.reduce_max(inputs, axis=-1, keepdims=True)

# =================== ENABLE MIXED PRECISION ===================
mixed_precision.set_global_policy('mixed_float16')

# =================== CONSTANTS ===================
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 4
EPOCHS = 60
train_ratio, val_ratio = 0.7, 0.15

# =================== DIRECTORIES ===================
dataset_dir = '/kaggle/input/colon-augmented-custom/colon_augmented_png_v9'
base_dir = '/kaggle/working/colon_split'

# =================== CLEAN EXISTING SPLITS ===================
if os.path.exists(base_dir):
    for folder in os.listdir(base_dir):
        rmtree(os.path.join(base_dir, folder))

# =================== DATA SPLIT ===================
splits = ['train', 'val', 'test']
class_names = os.listdir(dataset_dir)

for split in splits:
    for class_name in class_names:
        os.makedirs(os.path.join(base_dir, split, class_name), exist_ok=True)

for class_name in class_names:
    image_list = os.listdir(os.path.join(dataset_dir, class_name))
    random.shuffle(image_list)

    n_total = len(image_list)
    n_train = int(train_ratio * n_total)
    n_val = int(val_ratio * n_total)

    train_files = image_list[:n_train]
    val_files = image_list[n_train:n_train + n_val]
    test_files = image_list[n_train + n_val:]

    for split, split_files in zip(['train', 'val', 'test'], [train_files, val_files, test_files]):
        for img in tqdm(split_files, desc=f'{class_name} - {split}'):
            src = os.path.join(dataset_dir, class_name, img)
            dst = os.path.join(base_dir, split, class_name, img)
            copyfile(src, dst)

# =================== DATA GENERATORS ===================
train_datagen = ImageDataGenerator(
    rescale=1./255,
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input
)
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    os.path.join(base_dir, 'train'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_generator = val_test_datagen.flow_from_directory(
    os.path.join(base_dir, 'val'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_generator = val_test_datagen.flow_from_directory(
    os.path.join(base_dir, 'test'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# =================== MODEL BUILDING ===================
def cbam_block(input_feature, ratio=8):
    channel = input_feature.shape[-1]

    avg_pool = layers.GlobalAveragePooling2D()(input_feature)
    max_pool = layers.GlobalMaxPooling2D()(input_feature)

    shared_dense_one = layers.Dense(channel // ratio, activation='relu')
    shared_dense_two = layers.Dense(channel)

    mlp_avg = shared_dense_two(shared_dense_one(avg_pool))
    mlp_max = shared_dense_two(shared_dense_one(max_pool))

    channel_attention = layers.Add()([mlp_avg, mlp_max])
    channel_attention = layers.Activation('sigmoid')(channel_attention)
    channel_attention = layers.Reshape((1, 1, channel))(channel_attention)

    x = layers.Multiply()([input_feature, channel_attention])

    avg_pool_spatial = MeanChannel()(x)
    max_pool_spatial = MaxChannel()(x)
    concat = layers.Concatenate(axis=-1)([avg_pool_spatial, max_pool_spatial])
    spatial_attention = layers.Conv2D(1, kernel_size=7, padding='same', activation='sigmoid')(concat)

    refined_feature = layers.Multiply()([x, spatial_attention])
    return refined_feature

def build_model():
    base_model = tf.keras.applications.EfficientNetB0(
        include_top=False, input_shape=(224, 224, 3), weights='imagenet'
    )
    base_model.trainable = True

    inputs = layers.Input(shape=(224, 224, 3))
    x = base_model(inputs, training=True)
    x = cbam_block(x)

    x = layers.GlobalMaxPooling2D()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)

    model = tf.keras.Model(inputs, outputs)
    return model

model = build_model()

# =================== OPTIMIZER & LOSS ===================
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1.5e-3,
    decay_steps=1000,
    alpha=1e-6
)
optimizer = AdamW(learning_rate=lr_schedule, weight_decay=1e-5)
loss_fn = CategoricalCrossentropy(label_smoothing=0.1)

model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])

# =================== CALLBACKS ===================
checkpoint_cb = callbacks.ModelCheckpoint(
    'carenet.keras', monitor='val_accuracy', save_best_only=True, verbose=1
)
earlystop_cb = callbacks.EarlyStopping(
    monitor='val_accuracy', patience=7, restore_best_weights=True
)

# =================== TRAINING ===================
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=[checkpoint_cb, earlystop_cb],
    verbose=1
)

# =================== EVALUATION ===================
model.load_weights("carenet.keras")
test_loss, test_acc = model.evaluate(test_generator)
print(f"\n✅ Test Accuracy: {test_acc:.4f}")


1_ulcerative_colitis - test: 100%|██████████| 225/225 [00:00<00:00, 454.31it/s]


Found 4200 images belonging to 4 classes.
Found 900 images belonging to 4 classes.
Found 900 images belonging to 4 classes.
Epoch 1/60
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 432ms/step - accuracy: 0.4596 - loss: 2.4562
Epoch 1: val_accuracy improved from -inf to 0.16667, saving model to carenet.keras
132/132 ━━━━━━━━━━━━━━━━━━━━ 187s 560ms/step - accuracy: 0.4606 - loss: 2.4494 - val_accuracy: 0.1667 - val_loss: 1.4454
Epoch 2/60
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - accuracy: 0.8087 - loss: 0.7705
Epoch 2: val_accuracy improved from 0.16667 to 0.31444, saving model to carenet.keras
132/132 ━━━━━━━━━━━━━━━━━━━━ 15s 111ms/step - accuracy: 0.8088 - loss: 0.7702 - val_accuracy: 0.3144 - val_loss: 1.4055
Epoch 3/60
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - accuracy: 0.8898 - loss: 0.6104
Epoch 3: val_accuracy improved from 0.31444 to 0.57667, saving model to carenet.keras
132/132 ━━━━━━━━━━━━━━━━━━━━ 15s 111ms/step - accuracy: 0.8898 - loss: 0.6104 - val_accuracy: 0.5767 - val_loss: 1.116

# Classifier Head - Flatten

In [9]:
# =================== IMPORTS ===================
import os
import numpy as np
import random
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, backend as K
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from keras.optimizers import AdamW
from tensorflow.keras.losses import CategoricalCrossentropy
from shutil import rmtree, copyfile
from tqdm import tqdm
from tensorflow.keras import mixed_precision

# =================== CUSTOM LAYERS ===================
@tf.keras.utils.register_keras_serializable()
class MeanChannel(layers.Layer):
    def call(self, inputs):
        return tf.reduce_mean(inputs, axis=-1, keepdims=True)

@tf.keras.utils.register_keras_serializable()
class MaxChannel(layers.Layer):
    def call(self, inputs):
        return tf.reduce_max(inputs, axis=-1, keepdims=True)

# =================== ENABLE MIXED PRECISION ===================
mixed_precision.set_global_policy('mixed_float16')

# =================== CONSTANTS ===================
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 4
EPOCHS = 60
train_ratio, val_ratio = 0.7, 0.15

# =================== DIRECTORIES ===================
dataset_dir = '/kaggle/input/colon-augmented-custom/colon_augmented_png_v9'
base_dir = '/kaggle/working/colon_split'

# =================== CLEAN EXISTING SPLITS ===================
if os.path.exists(base_dir):
    for folder in os.listdir(base_dir):
        rmtree(os.path.join(base_dir, folder))

# =================== DATA SPLIT ===================
splits = ['train', 'val', 'test']
class_names = os.listdir(dataset_dir)

for split in splits:
    for class_name in class_names:
        os.makedirs(os.path.join(base_dir, split, class_name), exist_ok=True)

for class_name in class_names:
    image_list = os.listdir(os.path.join(dataset_dir, class_name))
    random.shuffle(image_list)

    n_total = len(image_list)
    n_train = int(train_ratio * n_total)
    n_val = int(val_ratio * n_total)

    train_files = image_list[:n_train]
    val_files = image_list[n_train:n_train + n_val]
    test_files = image_list[n_train + n_val:]

    for split, split_files in zip(['train', 'val', 'test'], [train_files, val_files, test_files]):
        for img in tqdm(split_files, desc=f'{class_name} - {split}'):
            src = os.path.join(dataset_dir, class_name, img)
            dst = os.path.join(base_dir, split, class_name, img)
            copyfile(src, dst)

# =================== DATA GENERATORS ===================
train_datagen = ImageDataGenerator(
    rescale=1./255,
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input
)
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    os.path.join(base_dir, 'train'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_generator = val_test_datagen.flow_from_directory(
    os.path.join(base_dir, 'val'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_generator = val_test_datagen.flow_from_directory(
    os.path.join(base_dir, 'test'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# =================== MODEL BUILDING ===================
def cbam_block(input_feature, ratio=8):
    channel = input_feature.shape[-1]

    avg_pool = layers.GlobalAveragePooling2D()(input_feature)
    max_pool = layers.GlobalMaxPooling2D()(input_feature)

    shared_dense_one = layers.Dense(channel // ratio, activation='relu')
    shared_dense_two = layers.Dense(channel)

    mlp_avg = shared_dense_two(shared_dense_one(avg_pool))
    mlp_max = shared_dense_two(shared_dense_one(max_pool))

    channel_attention = layers.Add()([mlp_avg, mlp_max])
    channel_attention = layers.Activation('sigmoid')(channel_attention)
    channel_attention = layers.Reshape((1, 1, channel))(channel_attention)

    x = layers.Multiply()([input_feature, channel_attention])

    avg_pool_spatial = MeanChannel()(x)
    max_pool_spatial = MaxChannel()(x)
    concat = layers.Concatenate(axis=-1)([avg_pool_spatial, max_pool_spatial])
    spatial_attention = layers.Conv2D(1, kernel_size=7, padding='same', activation='sigmoid')(concat)

    refined_feature = layers.Multiply()([x, spatial_attention])
    return refined_feature

def build_model():
    base_model = tf.keras.applications.EfficientNetB0(
        include_top=False, input_shape=(224, 224, 3), weights='imagenet'
    )
    base_model.trainable = True

    inputs = layers.Input(shape=(224, 224, 3))
    x = base_model(inputs, training=True)
    x = cbam_block(x)

    x = layers.Flatten()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)

    model = tf.keras.Model(inputs, outputs)
    return model

model = build_model()

# =================== OPTIMIZER & LOSS ===================
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1.5e-3,
    decay_steps=1000,
    alpha=1e-6
)
optimizer = AdamW(learning_rate=lr_schedule, weight_decay=1e-5)
loss_fn = CategoricalCrossentropy(label_smoothing=0.1)

model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])

# =================== CALLBACKS ===================
checkpoint_cb = callbacks.ModelCheckpoint(
    'carenet.keras', monitor='val_accuracy', save_best_only=True, verbose=1
)
earlystop_cb = callbacks.EarlyStopping(
    monitor='val_accuracy', patience=7, restore_best_weights=True
)

# =================== TRAINING ===================
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=[checkpoint_cb, earlystop_cb],
    verbose=1
)

# =================== EVALUATION ===================
model.load_weights("carenet.keras")
test_loss, test_acc = model.evaluate(test_generator)
print(f"\n✅ Test Accuracy: {test_acc:.4f}")


1_ulcerative_colitis - test: 100%|██████████| 225/225 [00:00<00:00, 418.37it/s]

Found 4200 images belonging to 4 classes.


Found 900 images belonging to 4 classes.
Found 900 images belonging to 4 classes.
Epoch 1/60
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 439ms/step - accuracy: 0.7418 - loss: 0.8984
Epoch 1: val_accuracy improved from -inf to 0.25000, saving model to carenet.keras
132/132 ━━━━━━━━━━━━━━━━━━━━ 196s 595ms/step - accuracy: 0.7427 - loss: 0.8966 - val_accuracy: 0.2500 - val_loss: 1.5725
Epoch 2/60
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - accuracy: 0.9713 - loss: 0.4507
Epoch 2: val_accuracy improved from 0.25000 to 0.48667, saving model to carenet.keras
132/132 ━━━━━━━━━━━━━━━━━━━━ 18s 139ms/step - accuracy: 0.9713 - loss: 0.4507 - val_accuracy: 0.4867 - val_loss: 1.2423
Epoch 3/60
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.9895 - loss: 0.4083
Epoch 3: val_accuracy improved from 0.48667 to 0.77333, saving model to carenet.keras
132/132 ━━━━━━━━━━━━━━━━━━━━ 18s 135ms/step - accuracy: 0.9895 - loss: 0.4083 - val_accuracy: 0.7733 - val_loss: 0.9324
Epoch 4/60
132/132 ━━━━━━━━━━━━━━━━━━━━ 

# *Optimizer*

# Optimizer - AdamW 

In [10]:
# =================== IMPORTS ===================
import os
import numpy as np
import random
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, backend as K
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from keras.optimizers import AdamW
from tensorflow.keras.losses import CategoricalCrossentropy
from shutil import rmtree, copyfile
from tqdm import tqdm
from tensorflow.keras import mixed_precision

# =================== CUSTOM LAYERS ===================
@tf.keras.utils.register_keras_serializable()
class MeanChannel(layers.Layer):
    def call(self, inputs):
        return tf.reduce_mean(inputs, axis=-1, keepdims=True)

@tf.keras.utils.register_keras_serializable()
class MaxChannel(layers.Layer):
    def call(self, inputs):
        return tf.reduce_max(inputs, axis=-1, keepdims=True)

# =================== ENABLE MIXED PRECISION ===================
mixed_precision.set_global_policy('mixed_float16')

# =================== CONSTANTS ===================
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 4
EPOCHS = 60
train_ratio, val_ratio = 0.7, 0.15

# =================== DIRECTORIES ===================
dataset_dir = '/kaggle/input/colon-augmented-custom/colon_augmented_png_v9'
base_dir = '/kaggle/working/colon_split'

# =================== CLEAN EXISTING SPLITS ===================
if os.path.exists(base_dir):
    for folder in os.listdir(base_dir):
        rmtree(os.path.join(base_dir, folder))

# =================== DATA SPLIT ===================
splits = ['train', 'val', 'test']
class_names = os.listdir(dataset_dir)

for split in splits:
    for class_name in class_names:
        os.makedirs(os.path.join(base_dir, split, class_name), exist_ok=True)

for class_name in class_names:
    image_list = os.listdir(os.path.join(dataset_dir, class_name))
    random.shuffle(image_list)

    n_total = len(image_list)
    n_train = int(train_ratio * n_total)
    n_val = int(val_ratio * n_total)

    train_files = image_list[:n_train]
    val_files = image_list[n_train:n_train + n_val]
    test_files = image_list[n_train + n_val:]

    for split, split_files in zip(['train', 'val', 'test'], [train_files, val_files, test_files]):
        for img in tqdm(split_files, desc=f'{class_name} - {split}'):
            src = os.path.join(dataset_dir, class_name, img)
            dst = os.path.join(base_dir, split, class_name, img)
            copyfile(src, dst)

# =================== DATA GENERATORS ===================
train_datagen = ImageDataGenerator(
    rescale=1./255,
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input
)
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    os.path.join(base_dir, 'train'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_generator = val_test_datagen.flow_from_directory(
    os.path.join(base_dir, 'val'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_generator = val_test_datagen.flow_from_directory(
    os.path.join(base_dir, 'test'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# =================== MODEL BUILDING ===================
def cbam_block(input_feature, ratio=8):
    channel = input_feature.shape[-1]

    avg_pool = layers.GlobalAveragePooling2D()(input_feature)
    max_pool = layers.GlobalMaxPooling2D()(input_feature)

    shared_dense_one = layers.Dense(channel // ratio, activation='relu')
    shared_dense_two = layers.Dense(channel)

    mlp_avg = shared_dense_two(shared_dense_one(avg_pool))
    mlp_max = shared_dense_two(shared_dense_one(max_pool))

    channel_attention = layers.Add()([mlp_avg, mlp_max])
    channel_attention = layers.Activation('sigmoid')(channel_attention)
    channel_attention = layers.Reshape((1, 1, channel))(channel_attention)

    x = layers.Multiply()([input_feature, channel_attention])

    avg_pool_spatial = MeanChannel()(x)
    max_pool_spatial = MaxChannel()(x)
    concat = layers.Concatenate(axis=-1)([avg_pool_spatial, max_pool_spatial])
    spatial_attention = layers.Conv2D(1, kernel_size=7, padding='same', activation='sigmoid')(concat)

    refined_feature = layers.Multiply()([x, spatial_attention])
    return refined_feature

def build_model():
    base_model = tf.keras.applications.EfficientNetB0(
        include_top=False, input_shape=(224, 224, 3), weights='imagenet'
    )
    base_model.trainable = True

    inputs = layers.Input(shape=(224, 224, 3))
    x = base_model(inputs, training=True)
    x = cbam_block(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)

    model = tf.keras.Model(inputs, outputs)
    return model

model = build_model()

# =================== OPTIMIZER & LOSS ===================
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1.5e-3,
    decay_steps=1000,
    alpha=1e-6
)
optimizer = AdamW(learning_rate=lr_schedule, weight_decay=1e-5)
loss_fn = CategoricalCrossentropy(label_smoothing=0.1)

model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])

# =================== CALLBACKS ===================
checkpoint_cb = callbacks.ModelCheckpoint(
    'carenet.keras', monitor='val_accuracy', save_best_only=True, verbose=1
)
earlystop_cb = callbacks.EarlyStopping(
    monitor='val_accuracy', patience=7, restore_best_weights=True
)

# =================== TRAINING ===================
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=[checkpoint_cb, earlystop_cb],
    verbose=1
)

# =================== EVALUATION ===================
model.load_weights("carenet.keras")
test_loss, test_acc = model.evaluate(test_generator)
print(f"\n✅ Test Accuracy: {test_acc:.4f}")


1_ulcerative_colitis - test: 100%|██████████| 225/225 [00:00<00:00, 447.02it/s]


Found 4200 images belonging to 4 classes.
Found 900 images belonging to 4 classes.
Found 900 images belonging to 4 classes.
Epoch 1/60
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 456ms/step - accuracy: 0.6132 - loss: 1.0406
Epoch 1: val_accuracy improved from -inf to 0.23889, saving model to carenet.keras
132/132 ━━━━━━━━━━━━━━━━━━━━ 192s 590ms/step - accuracy: 0.6144 - loss: 1.0387 - val_accuracy: 0.2389 - val_loss: 1.3926
Epoch 2/60
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - accuracy: 0.9315 - loss: 0.4958
Epoch 2: val_accuracy improved from 0.23889 to 0.40333, saving model to carenet.keras
132/132 ━━━━━━━━━━━━━━━━━━━━ 15s 111ms/step - accuracy: 0.9316 - loss: 0.4957 - val_accuracy: 0.4033 - val_loss: 1.3628
Epoch 3/60
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - accuracy: 0.9779 - loss: 0.4255
Epoch 3: val_accuracy improved from 0.40333 to 0.67222, saving model to carenet.keras
132/132 ━━━━━━━━━━━━━━━━━━━━ 15s 111ms/step - accuracy: 0.9778 - loss: 0.4255 - val_accuracy: 0.6722 - val_loss: 1.070

# Optimizer - Adam

In [2]:
# =================== IMPORTS ===================
import os
import numpy as np
import random
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, backend as K
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from keras.optimizers import AdamW
from tensorflow.keras.losses import CategoricalCrossentropy
from shutil import rmtree, copyfile
from tqdm import tqdm
from tensorflow.keras import mixed_precision
from tensorflow.keras.optimizers import Adam

# =================== CUSTOM LAYERS ===================
@tf.keras.utils.register_keras_serializable()
class MeanChannel(layers.Layer):
    def call(self, inputs):
        return tf.reduce_mean(inputs, axis=-1, keepdims=True)

@tf.keras.utils.register_keras_serializable()
class MaxChannel(layers.Layer):
    def call(self, inputs):
        return tf.reduce_max(inputs, axis=-1, keepdims=True)

# =================== ENABLE MIXED PRECISION ===================
mixed_precision.set_global_policy('mixed_float16')

# =================== CONSTANTS ===================
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 4
EPOCHS = 60
train_ratio, val_ratio = 0.7, 0.15

# =================== DIRECTORIES ===================
dataset_dir = '/kaggle/input/colon-augmented-custom/colon_augmented_png_v9'
base_dir = '/kaggle/working/colon_split'

# =================== CLEAN EXISTING SPLITS ===================
if os.path.exists(base_dir):
    for folder in os.listdir(base_dir):
        rmtree(os.path.join(base_dir, folder))

# =================== DATA SPLIT ===================
splits = ['train', 'val', 'test']
class_names = os.listdir(dataset_dir)

for split in splits:
    for class_name in class_names:
        os.makedirs(os.path.join(base_dir, split, class_name), exist_ok=True)

for class_name in class_names:
    image_list = os.listdir(os.path.join(dataset_dir, class_name))
    random.shuffle(image_list)

    n_total = len(image_list)
    n_train = int(train_ratio * n_total)
    n_val = int(val_ratio * n_total)

    train_files = image_list[:n_train]
    val_files = image_list[n_train:n_train + n_val]
    test_files = image_list[n_train + n_val:]

    for split, split_files in zip(['train', 'val', 'test'], [train_files, val_files, test_files]):
        for img in tqdm(split_files, desc=f'{class_name} - {split}'):
            src = os.path.join(dataset_dir, class_name, img)
            dst = os.path.join(base_dir, split, class_name, img)
            copyfile(src, dst)

# =================== DATA GENERATORS ===================
train_datagen = ImageDataGenerator(
    rescale=1./255,
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input
)
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    os.path.join(base_dir, 'train'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_generator = val_test_datagen.flow_from_directory(
    os.path.join(base_dir, 'val'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_generator = val_test_datagen.flow_from_directory(
    os.path.join(base_dir, 'test'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# =================== MODEL BUILDING ===================
def cbam_block(input_feature, ratio=8):
    channel = input_feature.shape[-1]

    avg_pool = layers.GlobalAveragePooling2D()(input_feature)
    max_pool = layers.GlobalMaxPooling2D()(input_feature)

    shared_dense_one = layers.Dense(channel // ratio, activation='relu')
    shared_dense_two = layers.Dense(channel)

    mlp_avg = shared_dense_two(shared_dense_one(avg_pool))
    mlp_max = shared_dense_two(shared_dense_one(max_pool))

    channel_attention = layers.Add()([mlp_avg, mlp_max])
    channel_attention = layers.Activation('sigmoid')(channel_attention)
    channel_attention = layers.Reshape((1, 1, channel))(channel_attention)

    x = layers.Multiply()([input_feature, channel_attention])

    avg_pool_spatial = MeanChannel()(x)
    max_pool_spatial = MaxChannel()(x)
    concat = layers.Concatenate(axis=-1)([avg_pool_spatial, max_pool_spatial])
    spatial_attention = layers.Conv2D(1, kernel_size=7, padding='same', activation='sigmoid')(concat)

    refined_feature = layers.Multiply()([x, spatial_attention])
    return refined_feature

def build_model():
    base_model = tf.keras.applications.EfficientNetB0(
        include_top=False, input_shape=(224, 224, 3), weights='imagenet'
    )
    base_model.trainable = True

    inputs = layers.Input(shape=(224, 224, 3))
    x = base_model(inputs, training=True)
    x = cbam_block(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)

    model = tf.keras.Model(inputs, outputs)
    return model

model = build_model()

# =================== OPTIMIZER & LOSS ===================
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1.5e-3,
    decay_steps=1000,
    alpha=1e-6
)

optimizer = Adam(learning_rate=lr_schedule)
loss_fn = CategoricalCrossentropy(label_smoothing=0.1)

model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])

# =================== CALLBACKS ===================
checkpoint_cb = callbacks.ModelCheckpoint(
    'carenet.keras', monitor='val_accuracy', save_best_only=True, verbose=1
)
earlystop_cb = callbacks.EarlyStopping(
    monitor='val_accuracy', patience=7, restore_best_weights=True
)

# =================== TRAINING ===================
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=[checkpoint_cb, earlystop_cb],
    verbose=1
)

# =================== EVALUATION ===================
model.load_weights("carenet.keras")
test_loss, test_acc = model.evaluate(test_generator)
print(f"\n✅ Test Accuracy: {test_acc:.4f}")


2025-07-25 14:30:08.120630: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753453808.488014      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753453808.590080      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
1_ulcerative_colitis - test: 100%|██████████| 225/225 [00:02<00:00, 92.86it/s]

Found 4200 images belonging to 4 classes.


Found 900 images belonging to 4 classes.
Found 900 images belonging to 4 classes.


I0000 00:00:1753453895.075781      36 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1753453895.076499      36 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/60


I0000 00:00:1753453961.067814     127 service.cc:148] XLA service 0x7a4804008010 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1753453961.069426     127 service.cc:156]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1753453961.069450     127 service.cc:156]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1753453967.576590     127 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1753454019.029329     127 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 476ms/step - accuracy: 0.6302 - loss: 1.0677
Epoch 1: val_accuracy improved from -inf to 0.25778, saving model to carenet.keras
132/132 ━━━━━━━━━━━━━━━━━━━━ 203s 624ms/step - accuracy: 0.6314 - loss: 1.0657 - val_accuracy: 0.2578 - val_loss: 1.4004
Epoch 2/60
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - accuracy: 0.9451 - loss: 0.4737
Epoch 2: val_accuracy improved from 0.25778 to 0.25889, saving model to carenet.keras
132/132 ━━━━━━━━━━━━━━━━━━━━ 15s 113ms/step - accuracy: 0.9451 - loss: 0.4737 - val_accuracy: 0.2589 - val_loss: 1.3687
Epoch 3/60
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - accuracy: 0.9678 - loss: 0.4286
Epoch 3: val_accuracy improved from 0.25889 to 0.62889, saving model to carenet.keras
132/132 ━━━━━━━━━━━━━━━━━━━━ 15s 114ms/step - accuracy: 0.9678 - loss: 0.4285 - val_accuracy: 0.6289 - val_loss: 1.0152
Epoch 4/60
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - accuracy: 0.9862 - loss: 0.3985
Epoch 4: val_accuracy improved from 0.62889 to

# Optimizer - Nadam

In [1]:
# =================== IMPORTS ===================
import os
import numpy as np
import random
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, backend as K
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from keras.optimizers import AdamW
from tensorflow.keras.losses import CategoricalCrossentropy
from shutil import rmtree, copyfile
from tqdm import tqdm
from tensorflow.keras import mixed_precision
from tensorflow.keras.optimizers import Nadam


# =================== CUSTOM LAYERS ===================
@tf.keras.utils.register_keras_serializable()
class MeanChannel(layers.Layer):
    def call(self, inputs):
        return tf.reduce_mean(inputs, axis=-1, keepdims=True)

@tf.keras.utils.register_keras_serializable()
class MaxChannel(layers.Layer):
    def call(self, inputs):
        return tf.reduce_max(inputs, axis=-1, keepdims=True)

# =================== ENABLE MIXED PRECISION ===================
mixed_precision.set_global_policy('mixed_float16')

# =================== CONSTANTS ===================
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 4
EPOCHS = 60
train_ratio, val_ratio = 0.7, 0.15

# =================== DIRECTORIES ===================
dataset_dir = '/kaggle/input/colon-augmented-custom/colon_augmented_png_v9'
base_dir = '/kaggle/working/colon_split'

# =================== CLEAN EXISTING SPLITS ===================
if os.path.exists(base_dir):
    for folder in os.listdir(base_dir):
        rmtree(os.path.join(base_dir, folder))

# =================== DATA SPLIT ===================
splits = ['train', 'val', 'test']
class_names = os.listdir(dataset_dir)

for split in splits:
    for class_name in class_names:
        os.makedirs(os.path.join(base_dir, split, class_name), exist_ok=True)

for class_name in class_names:
    image_list = os.listdir(os.path.join(dataset_dir, class_name))
    random.shuffle(image_list)

    n_total = len(image_list)
    n_train = int(train_ratio * n_total)
    n_val = int(val_ratio * n_total)

    train_files = image_list[:n_train]
    val_files = image_list[n_train:n_train + n_val]
    test_files = image_list[n_train + n_val:]

    for split, split_files in zip(['train', 'val', 'test'], [train_files, val_files, test_files]):
        for img in tqdm(split_files, desc=f'{class_name} - {split}'):
            src = os.path.join(dataset_dir, class_name, img)
            dst = os.path.join(base_dir, split, class_name, img)
            copyfile(src, dst)

# =================== DATA GENERATORS ===================
train_datagen = ImageDataGenerator(
    rescale=1./255,
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input
)
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    os.path.join(base_dir, 'train'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_generator = val_test_datagen.flow_from_directory(
    os.path.join(base_dir, 'val'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_generator = val_test_datagen.flow_from_directory(
    os.path.join(base_dir, 'test'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# =================== MODEL BUILDING ===================
def cbam_block(input_feature, ratio=8):
    channel = input_feature.shape[-1]

    avg_pool = layers.GlobalAveragePooling2D()(input_feature)
    max_pool = layers.GlobalMaxPooling2D()(input_feature)

    shared_dense_one = layers.Dense(channel // ratio, activation='relu')
    shared_dense_two = layers.Dense(channel)

    mlp_avg = shared_dense_two(shared_dense_one(avg_pool))
    mlp_max = shared_dense_two(shared_dense_one(max_pool))

    channel_attention = layers.Add()([mlp_avg, mlp_max])
    channel_attention = layers.Activation('sigmoid')(channel_attention)
    channel_attention = layers.Reshape((1, 1, channel))(channel_attention)

    x = layers.Multiply()([input_feature, channel_attention])

    avg_pool_spatial = MeanChannel()(x)
    max_pool_spatial = MaxChannel()(x)
    concat = layers.Concatenate(axis=-1)([avg_pool_spatial, max_pool_spatial])
    spatial_attention = layers.Conv2D(1, kernel_size=7, padding='same', activation='sigmoid')(concat)

    refined_feature = layers.Multiply()([x, spatial_attention])
    return refined_feature

def build_model():
    base_model = tf.keras.applications.EfficientNetB0(
        include_top=False, input_shape=(224, 224, 3), weights='imagenet'
    )
    base_model.trainable = True

    inputs = layers.Input(shape=(224, 224, 3))
    x = base_model(inputs, training=True)
    x = cbam_block(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)

    model = tf.keras.Model(inputs, outputs)
    return model

model = build_model()

# =================== OPTIMIZER & LOSS ===================
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1.5e-3,
    decay_steps=1000,
    alpha=1e-6
)

optimizer = Nadam(learning_rate=lr_schedule)
loss_fn = CategoricalCrossentropy(label_smoothing=0.1)

model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])

# =================== CALLBACKS ===================
checkpoint_cb = callbacks.ModelCheckpoint(
    'carenet.keras', monitor='val_accuracy', save_best_only=True, verbose=1
)
earlystop_cb = callbacks.EarlyStopping(
    monitor='val_accuracy', patience=7, restore_best_weights=True
)

# =================== TRAINING ===================
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=[checkpoint_cb, earlystop_cb],
    verbose=1
)

# =================== EVALUATION ===================
model.load_weights("carenet.keras")
test_loss, test_acc = model.evaluate(test_generator)
print(f"\n✅ Test Accuracy: {test_acc:.4f}")


2025-07-25 14:58:35.742737: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753455515.907311      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753455515.955309      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
1_ulcerative_colitis - test: 100%|██████████| 225/225 [00:01<00:00, 169.11it/s]


Found 4200 images belonging to 4 classes.
Found 900 images belonging to 4 classes.
Found 900 images belonging to 4 classes.


I0000 00:00:1753455566.455823      36 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1753455566.456497      36 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/60


I0000 00:00:1753455634.850763     126 service.cc:148] XLA service 0x7d4cf802d380 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1753455634.851642     126 service.cc:156]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1753455634.851663     126 service.cc:156]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1753455641.567788     126 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1753455693.425039     126 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 481ms/step - accuracy: 0.6265 - loss: 1.0143
Epoch 1: val_accuracy improved from -inf to 0.26333, saving model to carenet.keras
132/132 ━━━━━━━━━━━━━━━━━━━━ 206s 624ms/step - accuracy: 0.6277 - loss: 1.0124 - val_accuracy: 0.2633 - val_loss: 1.3926
Epoch 2/60
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - accuracy: 0.9340 - loss: 0.5003
Epoch 2: val_accuracy improved from 0.26333 to 0.48111, saving model to carenet.keras
132/132 ━━━━━━━━━━━━━━━━━━━━ 16s 117ms/step - accuracy: 0.9341 - loss: 0.5002 - val_accuracy: 0.4811 - val_loss: 1.2590
Epoch 3/60
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - accuracy: 0.9744 - loss: 0.4285
Epoch 3: val_accuracy improved from 0.48111 to 0.78889, saving model to carenet.keras
132/132 ━━━━━━━━━━━━━━━━━━━━ 15s 110ms/step - accuracy: 0.9744 - loss: 0.4285 - val_accuracy: 0.7889 - val_loss: 0.9576
Epoch 4/60
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step - accuracy: 0.9842 - loss: 0.4024
Epoch 4: val_accuracy improved from 0.78889 to

# *Learning Rate*

# Learning Rate - 1.5e-4 

In [5]:
# =================== IMPORTS ===================
import os
import numpy as np
import random
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, backend as K
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from keras.optimizers import AdamW
from tensorflow.keras.losses import CategoricalCrossentropy
from shutil import rmtree, copyfile
from tqdm import tqdm
from tensorflow.keras import mixed_precision

# =================== CUSTOM LAYERS ===================
@tf.keras.utils.register_keras_serializable()
class MeanChannel(layers.Layer):
    def call(self, inputs):
        return tf.reduce_mean(inputs, axis=-1, keepdims=True)

@tf.keras.utils.register_keras_serializable()
class MaxChannel(layers.Layer):
    def call(self, inputs):
        return tf.reduce_max(inputs, axis=-1, keepdims=True)

# =================== ENABLE MIXED PRECISION ===================
mixed_precision.set_global_policy('mixed_float16')

# =================== CONSTANTS ===================
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 4
EPOCHS = 60
train_ratio, val_ratio = 0.7, 0.15

# =================== DIRECTORIES ===================
dataset_dir = '/kaggle/input/colon-augmented-custom/colon_augmented_png_v9'
base_dir = '/kaggle/working/colon_split'

# =================== CLEAN EXISTING SPLITS ===================
if os.path.exists(base_dir):
    for folder in os.listdir(base_dir):
        rmtree(os.path.join(base_dir, folder))

# =================== DATA SPLIT ===================
splits = ['train', 'val', 'test']
class_names = os.listdir(dataset_dir)

for split in splits:
    for class_name in class_names:
        os.makedirs(os.path.join(base_dir, split, class_name), exist_ok=True)

for class_name in class_names:
    image_list = os.listdir(os.path.join(dataset_dir, class_name))
    random.shuffle(image_list)

    n_total = len(image_list)
    n_train = int(train_ratio * n_total)
    n_val = int(val_ratio * n_total)

    train_files = image_list[:n_train]
    val_files = image_list[n_train:n_train + n_val]
    test_files = image_list[n_train + n_val:]

    for split, split_files in zip(['train', 'val', 'test'], [train_files, val_files, test_files]):
        for img in tqdm(split_files, desc=f'{class_name} - {split}'):
            src = os.path.join(dataset_dir, class_name, img)
            dst = os.path.join(base_dir, split, class_name, img)
            copyfile(src, dst)

# =================== DATA GENERATORS ===================
train_datagen = ImageDataGenerator(
    rescale=1./255,
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input
)
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    os.path.join(base_dir, 'train'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_generator = val_test_datagen.flow_from_directory(
    os.path.join(base_dir, 'val'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_generator = val_test_datagen.flow_from_directory(
    os.path.join(base_dir, 'test'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# =================== MODEL BUILDING ===================
def cbam_block(input_feature, ratio=8):
    channel = input_feature.shape[-1]

    avg_pool = layers.GlobalAveragePooling2D()(input_feature)
    max_pool = layers.GlobalMaxPooling2D()(input_feature)

    shared_dense_one = layers.Dense(channel // ratio, activation='relu')
    shared_dense_two = layers.Dense(channel)

    mlp_avg = shared_dense_two(shared_dense_one(avg_pool))
    mlp_max = shared_dense_two(shared_dense_one(max_pool))

    channel_attention = layers.Add()([mlp_avg, mlp_max])
    channel_attention = layers.Activation('sigmoid')(channel_attention)
    channel_attention = layers.Reshape((1, 1, channel))(channel_attention)

    x = layers.Multiply()([input_feature, channel_attention])

    avg_pool_spatial = MeanChannel()(x)
    max_pool_spatial = MaxChannel()(x)
    concat = layers.Concatenate(axis=-1)([avg_pool_spatial, max_pool_spatial])
    spatial_attention = layers.Conv2D(1, kernel_size=7, padding='same', activation='sigmoid')(concat)

    refined_feature = layers.Multiply()([x, spatial_attention])
    return refined_feature

def build_model():
    base_model = tf.keras.applications.EfficientNetB0(
        include_top=False, input_shape=(224, 224, 3), weights='imagenet'
    )
    base_model.trainable = True

    inputs = layers.Input(shape=(224, 224, 3))
    x = base_model(inputs, training=True)
    x = cbam_block(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)

    model = tf.keras.Model(inputs, outputs)
    return model

model = build_model()

# =================== OPTIMIZER & LOSS ===================
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1.5e-4,
    decay_steps=1000,
    alpha=1e-6
)
optimizer = AdamW(learning_rate=lr_schedule, weight_decay=1e-5)
loss_fn = CategoricalCrossentropy(label_smoothing=0.1)

model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])

# =================== CALLBACKS ===================
checkpoint_cb = callbacks.ModelCheckpoint(
    'carenet.keras', monitor='val_accuracy', save_best_only=True, verbose=1
)
earlystop_cb = callbacks.EarlyStopping(
    monitor='val_accuracy', patience=7, restore_best_weights=True
)

# =================== TRAINING ===================
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=[checkpoint_cb, earlystop_cb],
    verbose=1
)

# =================== EVALUATION ===================
model.load_weights("carenet.keras")
test_loss, test_acc = model.evaluate(test_generator)
print(f"\n✅ Test Accuracy: {test_acc:.4f}")


1_ulcerative_colitis - test: 100%|██████████| 225/225 [00:00<00:00, 791.37it/s]


Found 4200 images belonging to 4 classes.
Found 900 images belonging to 4 classes.
Found 900 images belonging to 4 classes.
Epoch 1/60
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 415ms/step - accuracy: 0.6333 - loss: 1.0376
Epoch 1: val_accuracy improved from -inf to 0.26000, saving model to carenet.keras
132/132 ━━━━━━━━━━━━━━━━━━━━ 181s 540ms/step - accuracy: 0.6344 - loss: 1.0356 - val_accuracy: 0.2600 - val_loss: 1.3842
Epoch 2/60
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step - accuracy: 0.9483 - loss: 0.4754
Epoch 2: val_accuracy improved from 0.26000 to 0.27222, saving model to carenet.keras
132/132 ━━━━━━━━━━━━━━━━━━━━ 14s 109ms/step - accuracy: 0.9483 - loss: 0.4753 - val_accuracy: 0.2722 - val_loss: 1.4664
Epoch 3/60
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - accuracy: 0.9770 - loss: 0.4175
Epoch 3: val_accuracy improved from 0.27222 to 0.50111, saving model to carenet.keras
132/132 ━━━━━━━━━━━━━━━━━━━━ 14s 106ms/step - accuracy: 0.9770 - loss: 0.4175 - val_accuracy: 0.5011 - val_loss: 1.145

# Learning Rate - 1.5e-3

In [3]:
# =================== IMPORTS ===================
import os
import numpy as np
import random
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, backend as K
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from keras.optimizers import AdamW
from tensorflow.keras.losses import CategoricalCrossentropy
from shutil import rmtree, copyfile
from tqdm import tqdm
from tensorflow.keras import mixed_precision

# =================== CUSTOM LAYERS ===================
@tf.keras.utils.register_keras_serializable()
class MeanChannel(layers.Layer):
    def call(self, inputs):
        return tf.reduce_mean(inputs, axis=-1, keepdims=True)

@tf.keras.utils.register_keras_serializable()
class MaxChannel(layers.Layer):
    def call(self, inputs):
        return tf.reduce_max(inputs, axis=-1, keepdims=True)

# =================== ENABLE MIXED PRECISION ===================
mixed_precision.set_global_policy('mixed_float16')

# =================== CONSTANTS ===================
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 4
EPOCHS = 60
train_ratio, val_ratio = 0.7, 0.15

# =================== DIRECTORIES ===================
dataset_dir = '/kaggle/input/colon-augmented-custom/colon_augmented_png_v9'
base_dir = '/kaggle/working/colon_split'

# =================== CLEAN EXISTING SPLITS ===================
if os.path.exists(base_dir):
    for folder in os.listdir(base_dir):
        rmtree(os.path.join(base_dir, folder))

# =================== DATA SPLIT ===================
splits = ['train', 'val', 'test']
class_names = os.listdir(dataset_dir)

for split in splits:
    for class_name in class_names:
        os.makedirs(os.path.join(base_dir, split, class_name), exist_ok=True)

for class_name in class_names:
    image_list = os.listdir(os.path.join(dataset_dir, class_name))
    random.shuffle(image_list)

    n_total = len(image_list)
    n_train = int(train_ratio * n_total)
    n_val = int(val_ratio * n_total)

    train_files = image_list[:n_train]
    val_files = image_list[n_train:n_train + n_val]
    test_files = image_list[n_train + n_val:]

    for split, split_files in zip(['train', 'val', 'test'], [train_files, val_files, test_files]):
        for img in tqdm(split_files, desc=f'{class_name} - {split}'):
            src = os.path.join(dataset_dir, class_name, img)
            dst = os.path.join(base_dir, split, class_name, img)
            copyfile(src, dst)

# =================== DATA GENERATORS ===================
train_datagen = ImageDataGenerator(
    rescale=1./255,
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input
)
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    os.path.join(base_dir, 'train'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_generator = val_test_datagen.flow_from_directory(
    os.path.join(base_dir, 'val'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_generator = val_test_datagen.flow_from_directory(
    os.path.join(base_dir, 'test'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# =================== MODEL BUILDING ===================
def cbam_block(input_feature, ratio=8):
    channel = input_feature.shape[-1]

    avg_pool = layers.GlobalAveragePooling2D()(input_feature)
    max_pool = layers.GlobalMaxPooling2D()(input_feature)

    shared_dense_one = layers.Dense(channel // ratio, activation='relu')
    shared_dense_two = layers.Dense(channel)

    mlp_avg = shared_dense_two(shared_dense_one(avg_pool))
    mlp_max = shared_dense_two(shared_dense_one(max_pool))

    channel_attention = layers.Add()([mlp_avg, mlp_max])
    channel_attention = layers.Activation('sigmoid')(channel_attention)
    channel_attention = layers.Reshape((1, 1, channel))(channel_attention)

    x = layers.Multiply()([input_feature, channel_attention])

    avg_pool_spatial = MeanChannel()(x)
    max_pool_spatial = MaxChannel()(x)
    concat = layers.Concatenate(axis=-1)([avg_pool_spatial, max_pool_spatial])
    spatial_attention = layers.Conv2D(1, kernel_size=7, padding='same', activation='sigmoid')(concat)

    refined_feature = layers.Multiply()([x, spatial_attention])
    return refined_feature

def build_model():
    base_model = tf.keras.applications.EfficientNetB0(
        include_top=False, input_shape=(224, 224, 3), weights='imagenet'
    )
    base_model.trainable = True

    inputs = layers.Input(shape=(224, 224, 3))
    x = base_model(inputs, training=True)
    x = cbam_block(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)

    model = tf.keras.Model(inputs, outputs)
    return model

model = build_model()

# =================== OPTIMIZER & LOSS ===================
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1.5e-3,
    decay_steps=1000,
    alpha=1e-6
)
optimizer = AdamW(learning_rate=lr_schedule, weight_decay=1e-5)
loss_fn = CategoricalCrossentropy(label_smoothing=0.1)

model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])

# =================== CALLBACKS ===================
checkpoint_cb = callbacks.ModelCheckpoint(
    'carenet.keras', monitor='val_accuracy', save_best_only=True, verbose=1
)
earlystop_cb = callbacks.EarlyStopping(
    monitor='val_accuracy', patience=7, restore_best_weights=True
)

# =================== TRAINING ===================
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=[checkpoint_cb, earlystop_cb],
    verbose=1
)

# =================== EVALUATION ===================
model.load_weights("carenet.keras")
test_loss, test_acc = model.evaluate(test_generator)
print(f"\n✅ Test Accuracy: {test_acc:.4f}")


1_ulcerative_colitis - test: 100%|██████████| 225/225 [00:00<00:00, 510.38it/s]

Found 4200 images belonging to 4 classes.


Found 900 images belonging to 4 classes.
Found 900 images belonging to 4 classes.
Epoch 1/60
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 456ms/step - accuracy: 0.8045 - loss: 0.7337
Epoch 1: val_accuracy improved from -inf to 0.25444, saving model to carenet.keras
132/132 ━━━━━━━━━━━━━━━━━━━━ 198s 589ms/step - accuracy: 0.8052 - loss: 0.7325 - val_accuracy: 0.2544 - val_loss: 1.7561
Epoch 2/60
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step - accuracy: 0.9612 - loss: 0.4376
Epoch 2: val_accuracy improved from 0.25444 to 0.40000, saving model to carenet.keras
132/132 ━━━━━━━━━━━━━━━━━━━━ 16s 120ms/step - accuracy: 0.9612 - loss: 0.4376 - val_accuracy: 0.4000 - val_loss: 1.7431
Epoch 3/60
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - accuracy: 0.9833 - loss: 0.3935
Epoch 3: val_accuracy did not improve from 0.40000
132/132 ━━━━━━━━━━━━━━━━━━━━ 13s 101ms/step - accuracy: 0.9832 - loss: 0.3935 - val_accuracy: 0.2389 - val_loss: 1.9035
Epoch 4/60
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step - accuracy: 0.9899 - l

# Learning Rate - 1.5e-5

In [3]:
# =================== IMPORTS ===================
import os
import numpy as np
import random
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, backend as K
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from keras.optimizers import AdamW
from tensorflow.keras.losses import CategoricalCrossentropy
from shutil import rmtree, copyfile
from tqdm import tqdm
from tensorflow.keras import mixed_precision

# =================== CUSTOM LAYERS ===================
@tf.keras.utils.register_keras_serializable()
class MeanChannel(layers.Layer):
    def call(self, inputs):
        return tf.reduce_mean(inputs, axis=-1, keepdims=True)

@tf.keras.utils.register_keras_serializable()
class MaxChannel(layers.Layer):
    def call(self, inputs):
        return tf.reduce_max(inputs, axis=-1, keepdims=True)

# =================== ENABLE MIXED PRECISION ===================
mixed_precision.set_global_policy('mixed_float16')

# =================== CONSTANTS ===================
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 4
EPOCHS = 60
train_ratio, val_ratio = 0.7, 0.15

# =================== DIRECTORIES ===================
dataset_dir = '/kaggle/input/colon-augmented-custom/colon_augmented_png_v9'
base_dir = '/kaggle/working/colon_split'

# =================== CLEAN EXISTING SPLITS ===================
if os.path.exists(base_dir):
    for folder in os.listdir(base_dir):
        rmtree(os.path.join(base_dir, folder))

# =================== DATA SPLIT ===================
splits = ['train', 'val', 'test']
class_names = os.listdir(dataset_dir)

for split in splits:
    for class_name in class_names:
        os.makedirs(os.path.join(base_dir, split, class_name), exist_ok=True)

for class_name in class_names:
    image_list = os.listdir(os.path.join(dataset_dir, class_name))
    random.shuffle(image_list)

    n_total = len(image_list)
    n_train = int(train_ratio * n_total)
    n_val = int(val_ratio * n_total)

    train_files = image_list[:n_train]
    val_files = image_list[n_train:n_train + n_val]
    test_files = image_list[n_train + n_val:]

    for split, split_files in zip(['train', 'val', 'test'], [train_files, val_files, test_files]):
        for img in tqdm(split_files, desc=f'{class_name} - {split}'):
            src = os.path.join(dataset_dir, class_name, img)
            dst = os.path.join(base_dir, split, class_name, img)
            copyfile(src, dst)

# =================== DATA GENERATORS ===================
train_datagen = ImageDataGenerator(
    rescale=1./255,
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input
)
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    os.path.join(base_dir, 'train'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_generator = val_test_datagen.flow_from_directory(
    os.path.join(base_dir, 'val'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_generator = val_test_datagen.flow_from_directory(
    os.path.join(base_dir, 'test'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# =================== MODEL BUILDING ===================
def cbam_block(input_feature, ratio=8):
    channel = input_feature.shape[-1]

    avg_pool = layers.GlobalAveragePooling2D()(input_feature)
    max_pool = layers.GlobalMaxPooling2D()(input_feature)

    shared_dense_one = layers.Dense(channel // ratio, activation='relu')
    shared_dense_two = layers.Dense(channel)

    mlp_avg = shared_dense_two(shared_dense_one(avg_pool))
    mlp_max = shared_dense_two(shared_dense_one(max_pool))

    channel_attention = layers.Add()([mlp_avg, mlp_max])
    channel_attention = layers.Activation('sigmoid')(channel_attention)
    channel_attention = layers.Reshape((1, 1, channel))(channel_attention)

    x = layers.Multiply()([input_feature, channel_attention])

    avg_pool_spatial = MeanChannel()(x)
    max_pool_spatial = MaxChannel()(x)
    concat = layers.Concatenate(axis=-1)([avg_pool_spatial, max_pool_spatial])
    spatial_attention = layers.Conv2D(1, kernel_size=7, padding='same', activation='sigmoid')(concat)

    refined_feature = layers.Multiply()([x, spatial_attention])
    return refined_feature

def build_model():
    base_model = tf.keras.applications.EfficientNetB0(
        include_top=False, input_shape=(224, 224, 3), weights='imagenet'
    )
    base_model.trainable = True

    inputs = layers.Input(shape=(224, 224, 3))
    x = base_model(inputs, training=True)
    x = cbam_block(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)

    model = tf.keras.Model(inputs, outputs)
    return model

model = build_model()

# =================== OPTIMIZER & LOSS ===================
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1.5e-5,
    decay_steps=1000,
    alpha=1e-6
)
optimizer = AdamW(learning_rate=lr_schedule, weight_decay=1e-5)
loss_fn = CategoricalCrossentropy(label_smoothing=0.1)

model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])

# =================== CALLBACKS ===================
checkpoint_cb = callbacks.ModelCheckpoint(
    'carenet.keras', monitor='val_accuracy', save_best_only=True, verbose=1
)
earlystop_cb = callbacks.EarlyStopping(
    monitor='val_accuracy', patience=7, restore_best_weights=True
)

# =================== TRAINING ===================
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=[checkpoint_cb, earlystop_cb],
    verbose=1
)

# =================== EVALUATION ===================
model.load_weights("carenet.keras")
test_loss, test_acc = model.evaluate(test_generator)
print(f"\n✅ Test Accuracy: {test_acc:.4f}")


1_ulcerative_colitis - test: 100%|██████████| 225/225 [00:00<00:00, 804.50it/s]


Found 4200 images belonging to 4 classes.
Found 900 images belonging to 4 classes.
Found 900 images belonging to 4 classes.
Epoch 1/60
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 415ms/step - accuracy: 0.3405 - loss: 1.3651
Epoch 1: val_accuracy improved from -inf to 0.37889, saving model to carenet.keras
132/132 ━━━━━━━━━━━━━━━━━━━━ 184s 539ms/step - accuracy: 0.3410 - loss: 1.3649 - val_accuracy: 0.3789 - val_loss: 1.3850
Epoch 2/60
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - accuracy: 0.6283 - loss: 1.2128
Epoch 2: val_accuracy did not improve from 0.37889
132/132 ━━━━━━━━━━━━━━━━━━━━ 13s 99ms/step - accuracy: 0.6287 - loss: 1.2123 - val_accuracy: 0.3100 - val_loss: 1.3831
Epoch 3/60
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step - accuracy: 0.7567 - loss: 0.9268
Epoch 3: val_accuracy did not improve from 0.37889
132/132 ━━━━━━━━━━━━━━━━━━━━ 13s 97ms/step - accuracy: 0.7569 - loss: 0.9264 - val_accuracy: 0.3711 - val_loss: 1.3225
Epoch 4/60
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - accuracy: 0.822

# *Activation Function*

# Activation Function - Relu

In [8]:
# =================== IMPORTS ===================
import os
import numpy as np
import random
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, backend as K
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from keras.optimizers import AdamW
from tensorflow.keras.losses import CategoricalCrossentropy
from shutil import rmtree, copyfile
from tqdm import tqdm
from tensorflow.keras import mixed_precision

# =================== CUSTOM LAYERS ===================
@tf.keras.utils.register_keras_serializable()
class MeanChannel(layers.Layer):
    def call(self, inputs):
        return tf.reduce_mean(inputs, axis=-1, keepdims=True)

@tf.keras.utils.register_keras_serializable()
class MaxChannel(layers.Layer):
    def call(self, inputs):
        return tf.reduce_max(inputs, axis=-1, keepdims=True)

# =================== ENABLE MIXED PRECISION ===================
mixed_precision.set_global_policy('mixed_float16')

# =================== CONSTANTS ===================
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 4
EPOCHS = 60
train_ratio, val_ratio = 0.7, 0.15

# =================== DIRECTORIES ===================
dataset_dir = '/kaggle/input/colon-augmented-custom/colon_augmented_png_v9'
base_dir = '/kaggle/working/colon_split'

# =================== CLEAN EXISTING SPLITS ===================
if os.path.exists(base_dir):
    for folder in os.listdir(base_dir):
        rmtree(os.path.join(base_dir, folder))

# =================== DATA SPLIT ===================
splits = ['train', 'val', 'test']
class_names = os.listdir(dataset_dir)

for split in splits:
    for class_name in class_names:
        os.makedirs(os.path.join(base_dir, split, class_name), exist_ok=True)

for class_name in class_names:
    image_list = os.listdir(os.path.join(dataset_dir, class_name))
    random.shuffle(image_list)

    n_total = len(image_list)
    n_train = int(train_ratio * n_total)
    n_val = int(val_ratio * n_total)

    train_files = image_list[:n_train]
    val_files = image_list[n_train:n_train + n_val]
    test_files = image_list[n_train + n_val:]

    for split, split_files in zip(['train', 'val', 'test'], [train_files, val_files, test_files]):
        for img in tqdm(split_files, desc=f'{class_name} - {split}'):
            src = os.path.join(dataset_dir, class_name, img)
            dst = os.path.join(base_dir, split, class_name, img)
            copyfile(src, dst)

# =================== DATA GENERATORS ===================
train_datagen = ImageDataGenerator(
    rescale=1./255,
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input
)
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    os.path.join(base_dir, 'train'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_generator = val_test_datagen.flow_from_directory(
    os.path.join(base_dir, 'val'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_generator = val_test_datagen.flow_from_directory(
    os.path.join(base_dir, 'test'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# =================== MODEL BUILDING ===================
def cbam_block(input_feature, ratio=8):
    channel = input_feature.shape[-1]

    avg_pool = layers.GlobalAveragePooling2D()(input_feature)
    max_pool = layers.GlobalMaxPooling2D()(input_feature)

    shared_dense_one = layers.Dense(channel // ratio, activation='relu')
    shared_dense_two = layers.Dense(channel)

    mlp_avg = shared_dense_two(shared_dense_one(avg_pool))
    mlp_max = shared_dense_two(shared_dense_one(max_pool))

    channel_attention = layers.Add()([mlp_avg, mlp_max])
    channel_attention = layers.Activation('sigmoid')(channel_attention)
    channel_attention = layers.Reshape((1, 1, channel))(channel_attention)

    x = layers.Multiply()([input_feature, channel_attention])

    avg_pool_spatial = MeanChannel()(x)
    max_pool_spatial = MaxChannel()(x)
    concat = layers.Concatenate(axis=-1)([avg_pool_spatial, max_pool_spatial])
    spatial_attention = layers.Conv2D(1, kernel_size=7, padding='same', activation='sigmoid')(concat)

    refined_feature = layers.Multiply()([x, spatial_attention])
    return refined_feature

def build_model():
    base_model = tf.keras.applications.EfficientNetB0(
        include_top=False, input_shape=(224, 224, 3), weights='imagenet'
    )
    base_model.trainable = True

    inputs = layers.Input(shape=(224, 224, 3))
    x = base_model(inputs, training=True)
    x = cbam_block(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)

    model = tf.keras.Model(inputs, outputs)
    return model

model = build_model()

# =================== OPTIMIZER & LOSS ===================
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1.5e-3,
    decay_steps=1000,
    alpha=1e-6
)
optimizer = AdamW(learning_rate=lr_schedule, weight_decay=1e-5)
loss_fn = CategoricalCrossentropy(label_smoothing=0.1)

model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])

# =================== CALLBACKS ===================
checkpoint_cb = callbacks.ModelCheckpoint(
    'carenet.keras', monitor='val_accuracy', save_best_only=True, verbose=1
)
earlystop_cb = callbacks.EarlyStopping(
    monitor='val_accuracy', patience=7, restore_best_weights=True
)

# =================== TRAINING ===================
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=[checkpoint_cb, earlystop_cb],
    verbose=1
)

# =================== EVALUATION ===================
model.load_weights("carenet.keras")
test_loss, test_acc = model.evaluate(test_generator)
print(f"\n✅ Test Accuracy: {test_acc:.4f}")


1_ulcerative_colitis - test: 100%|██████████| 225/225 [00:00<00:00, 464.87it/s]


Found 4200 images belonging to 4 classes.
Found 900 images belonging to 4 classes.
Found 900 images belonging to 4 classes.
Epoch 1/60
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 412ms/step - accuracy: 0.6364 - loss: 1.0037
Epoch 1: val_accuracy improved from -inf to 0.25000, saving model to carenet.keras
132/132 ━━━━━━━━━━━━━━━━━━━━ 178s 537ms/step - accuracy: 0.6376 - loss: 1.0018 - val_accuracy: 0.2500 - val_loss: 1.3992
Epoch 2/60
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - accuracy: 0.9487 - loss: 0.4774
Epoch 2: val_accuracy improved from 0.25000 to 0.43667, saving model to carenet.keras
132/132 ━━━━━━━━━━━━━━━━━━━━ 14s 106ms/step - accuracy: 0.9487 - loss: 0.4774 - val_accuracy: 0.4367 - val_loss: 1.3219
Epoch 3/60
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.9691 - loss: 0.4312
Epoch 3: val_accuracy improved from 0.43667 to 0.46333, saving model to carenet.keras
132/132 ━━━━━━━━━━━━━━━━━━━━ 14s 106ms/step - accuracy: 0.9692 - loss: 0.4312 - val_accuracy: 0.4633 - val_loss: 1.242

# Activation Function - Gelu

In [9]:
# =================== IMPORTS ===================
import os
import numpy as np
import random
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, backend as K
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from keras.optimizers import AdamW
from tensorflow.keras.losses import CategoricalCrossentropy
from shutil import rmtree, copyfile
from tqdm import tqdm
from tensorflow.keras import mixed_precision

# =================== CUSTOM LAYERS ===================
@tf.keras.utils.register_keras_serializable()
class MeanChannel(layers.Layer):
    def call(self, inputs):
        return tf.reduce_mean(inputs, axis=-1, keepdims=True)

@tf.keras.utils.register_keras_serializable()
class MaxChannel(layers.Layer):
    def call(self, inputs):
        return tf.reduce_max(inputs, axis=-1, keepdims=True)

# =================== ENABLE MIXED PRECISION ===================
mixed_precision.set_global_policy('mixed_float16')

# =================== CONSTANTS ===================
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 4
EPOCHS = 60
train_ratio, val_ratio = 0.7, 0.15

# =================== DIRECTORIES ===================
dataset_dir = '/kaggle/input/colon-augmented-custom/colon_augmented_png_v9'
base_dir = '/kaggle/working/colon_split'

# =================== CLEAN EXISTING SPLITS ===================
if os.path.exists(base_dir):
    for folder in os.listdir(base_dir):
        rmtree(os.path.join(base_dir, folder))

# =================== DATA SPLIT ===================
splits = ['train', 'val', 'test']
class_names = os.listdir(dataset_dir)

for split in splits:
    for class_name in class_names:
        os.makedirs(os.path.join(base_dir, split, class_name), exist_ok=True)

for class_name in class_names:
    image_list = os.listdir(os.path.join(dataset_dir, class_name))
    random.shuffle(image_list)

    n_total = len(image_list)
    n_train = int(train_ratio * n_total)
    n_val = int(val_ratio * n_total)

    train_files = image_list[:n_train]
    val_files = image_list[n_train:n_train + n_val]
    test_files = image_list[n_train + n_val:]

    for split, split_files in zip(['train', 'val', 'test'], [train_files, val_files, test_files]):
        for img in tqdm(split_files, desc=f'{class_name} - {split}'):
            src = os.path.join(dataset_dir, class_name, img)
            dst = os.path.join(base_dir, split, class_name, img)
            copyfile(src, dst)

# =================== DATA GENERATORS ===================
train_datagen = ImageDataGenerator(
    rescale=1./255,
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input
)
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    os.path.join(base_dir, 'train'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_generator = val_test_datagen.flow_from_directory(
    os.path.join(base_dir, 'val'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_generator = val_test_datagen.flow_from_directory(
    os.path.join(base_dir, 'test'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# =================== MODEL BUILDING ===================
def cbam_block(input_feature, ratio=8):
    channel = input_feature.shape[-1]

    avg_pool = layers.GlobalAveragePooling2D()(input_feature)
    max_pool = layers.GlobalMaxPooling2D()(input_feature)

    shared_dense_one = layers.Dense(channel // ratio, activation='relu')
    shared_dense_two = layers.Dense(channel)

    mlp_avg = shared_dense_two(shared_dense_one(avg_pool))
    mlp_max = shared_dense_two(shared_dense_one(max_pool))

    channel_attention = layers.Add()([mlp_avg, mlp_max])
    channel_attention = layers.Activation('sigmoid')(channel_attention)
    channel_attention = layers.Reshape((1, 1, channel))(channel_attention)

    x = layers.Multiply()([input_feature, channel_attention])

    avg_pool_spatial = MeanChannel()(x)
    max_pool_spatial = MaxChannel()(x)
    concat = layers.Concatenate(axis=-1)([avg_pool_spatial, max_pool_spatial])
    spatial_attention = layers.Conv2D(1, kernel_size=7, padding='same', activation='sigmoid')(concat)

    refined_feature = layers.Multiply()([x, spatial_attention])
    return refined_feature

def build_model():
    base_model = tf.keras.applications.EfficientNetB0(
        include_top=False, input_shape=(224, 224, 3), weights='imagenet'
    )
    base_model.trainable = True

    inputs = layers.Input(shape=(224, 224, 3))
    x = base_model(inputs, training=True)
    x = cbam_block(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(512, activation='gelu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)

    model = tf.keras.Model(inputs, outputs)
    return model

model = build_model()

# =================== OPTIMIZER & LOSS ===================
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1.5e-3,
    decay_steps=1000,
    alpha=1e-6
)
optimizer = AdamW(learning_rate=lr_schedule, weight_decay=1e-5)
loss_fn = CategoricalCrossentropy(label_smoothing=0.1)

model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])

# =================== CALLBACKS ===================
checkpoint_cb = callbacks.ModelCheckpoint(
    'carenet.keras', monitor='val_accuracy', save_best_only=True, verbose=1
)
earlystop_cb = callbacks.EarlyStopping(
    monitor='val_accuracy', patience=7, restore_best_weights=True
)

# =================== TRAINING ===================
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=[checkpoint_cb, earlystop_cb],
    verbose=1
)

# =================== EVALUATION ===================
model.load_weights("carenet.keras")
test_loss, test_acc = model.evaluate(test_generator)
print(f"\n✅ Test Accuracy: {test_acc:.4f}")


1_ulcerative_colitis - test: 100%|██████████| 225/225 [00:00<00:00, 824.99it/s]


Found 4200 images belonging to 4 classes.
Found 900 images belonging to 4 classes.
Found 900 images belonging to 4 classes.
Epoch 1/60
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 421ms/step - accuracy: 0.6598 - loss: 0.9939
Epoch 1: val_accuracy improved from -inf to 0.30333, saving model to carenet.keras
132/132 ━━━━━━━━━━━━━━━━━━━━ 186s 549ms/step - accuracy: 0.6609 - loss: 0.9920 - val_accuracy: 0.3033 - val_loss: 1.3844
Epoch 2/60
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - accuracy: 0.9537 - loss: 0.4609
Epoch 2: val_accuracy improved from 0.30333 to 0.44000, saving model to carenet.keras
132/132 ━━━━━━━━━━━━━━━━━━━━ 14s 109ms/step - accuracy: 0.9538 - loss: 0.4609 - val_accuracy: 0.4400 - val_loss: 1.2100
Epoch 3/60
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - accuracy: 0.9797 - loss: 0.4108
Epoch 3: val_accuracy improved from 0.44000 to 0.50889, saving model to carenet.keras
132/132 ━━━━━━━━━━━━━━━━━━━━ 14s 106ms/step - accuracy: 0.9797 - loss: 0.4108 - val_accuracy: 0.5089 - val_loss: 1.090

# Activation Function - Swish

In [11]:
# =================== IMPORTS ===================
import os
import numpy as np
import random
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, backend as K
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from keras.optimizers import AdamW
from tensorflow.keras.losses import CategoricalCrossentropy
from shutil import rmtree, copyfile
from tqdm import tqdm
from tensorflow.keras import mixed_precision

# =================== CUSTOM LAYERS ===================
@tf.keras.utils.register_keras_serializable()
class MeanChannel(layers.Layer):
    def call(self, inputs):
        return tf.reduce_mean(inputs, axis=-1, keepdims=True)

@tf.keras.utils.register_keras_serializable()
class MaxChannel(layers.Layer):
    def call(self, inputs):
        return tf.reduce_max(inputs, axis=-1, keepdims=True)

# =================== ENABLE MIXED PRECISION ===================
mixed_precision.set_global_policy('mixed_float16')

# =================== CONSTANTS ===================
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 4
EPOCHS = 60
train_ratio, val_ratio = 0.7, 0.15

# =================== DIRECTORIES ===================
dataset_dir = '/kaggle/input/colon-augmented-custom/colon_augmented_png_v9'
base_dir = '/kaggle/working/colon_split'

# =================== CLEAN EXISTING SPLITS ===================
if os.path.exists(base_dir):
    for folder in os.listdir(base_dir):
        rmtree(os.path.join(base_dir, folder))

# =================== DATA SPLIT ===================
splits = ['train', 'val', 'test']
class_names = os.listdir(dataset_dir)

for split in splits:
    for class_name in class_names:
        os.makedirs(os.path.join(base_dir, split, class_name), exist_ok=True)

for class_name in class_names:
    image_list = os.listdir(os.path.join(dataset_dir, class_name))
    random.shuffle(image_list)

    n_total = len(image_list)
    n_train = int(train_ratio * n_total)
    n_val = int(val_ratio * n_total)

    train_files = image_list[:n_train]
    val_files = image_list[n_train:n_train + n_val]
    test_files = image_list[n_train + n_val:]

    for split, split_files in zip(['train', 'val', 'test'], [train_files, val_files, test_files]):
        for img in tqdm(split_files, desc=f'{class_name} - {split}'):
            src = os.path.join(dataset_dir, class_name, img)
            dst = os.path.join(base_dir, split, class_name, img)
            copyfile(src, dst)

# =================== DATA GENERATORS ===================
train_datagen = ImageDataGenerator(
    rescale=1./255,
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input
)
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    os.path.join(base_dir, 'train'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_generator = val_test_datagen.flow_from_directory(
    os.path.join(base_dir, 'val'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_generator = val_test_datagen.flow_from_directory(
    os.path.join(base_dir, 'test'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# =================== MODEL BUILDING ===================
def cbam_block(input_feature, ratio=8):
    channel = input_feature.shape[-1]

    avg_pool = layers.GlobalAveragePooling2D()(input_feature)
    max_pool = layers.GlobalMaxPooling2D()(input_feature)

    shared_dense_one = layers.Dense(channel // ratio, activation='relu')
    shared_dense_two = layers.Dense(channel)

    mlp_avg = shared_dense_two(shared_dense_one(avg_pool))
    mlp_max = shared_dense_two(shared_dense_one(max_pool))

    channel_attention = layers.Add()([mlp_avg, mlp_max])
    channel_attention = layers.Activation('sigmoid')(channel_attention)
    channel_attention = layers.Reshape((1, 1, channel))(channel_attention)

    x = layers.Multiply()([input_feature, channel_attention])

    avg_pool_spatial = MeanChannel()(x)
    max_pool_spatial = MaxChannel()(x)
    concat = layers.Concatenate(axis=-1)([avg_pool_spatial, max_pool_spatial])
    spatial_attention = layers.Conv2D(1, kernel_size=7, padding='same', activation='sigmoid')(concat)

    refined_feature = layers.Multiply()([x, spatial_attention])
    return refined_feature

def build_model():
    base_model = tf.keras.applications.EfficientNetB0(
        include_top=False, input_shape=(224, 224, 3), weights='imagenet'
    )
    base_model.trainable = True

    inputs = layers.Input(shape=(224, 224, 3))
    x = base_model(inputs, training=True)
    x = cbam_block(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(512, activation='swish')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)

    model = tf.keras.Model(inputs, outputs)
    return model

model = build_model()

# =================== OPTIMIZER & LOSS ===================
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1.5e-3,
    decay_steps=1000,
    alpha=1e-6
)
optimizer = AdamW(learning_rate=lr_schedule, weight_decay=1e-5)
loss_fn = CategoricalCrossentropy(label_smoothing=0.1)

model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])

# =================== CALLBACKS ===================
checkpoint_cb = callbacks.ModelCheckpoint(
    'carenet.keras', monitor='val_accuracy', save_best_only=True, verbose=1
)
earlystop_cb = callbacks.EarlyStopping(
    monitor='val_accuracy', patience=7, restore_best_weights=True
)

# =================== TRAINING ===================
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=[checkpoint_cb, earlystop_cb],
    verbose=1
)

# =================== EVALUATION ===================
model.load_weights("carenet.keras")
test_loss, test_acc = model.evaluate(test_generator)
print(f"\n✅ Test Accuracy: {test_acc:.4f}")


1_ulcerative_colitis - test: 100%|██████████| 225/225 [00:00<00:00, 417.89it/s]

Found 4200 images belonging to 4 classes.


Found 900 images belonging to 4 classes.
Found 900 images belonging to 4 classes.
Epoch 1/60
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 471ms/step - accuracy: 0.6591 - loss: 0.9520
Epoch 1: val_accuracy improved from -inf to 0.25000, saving model to carenet.keras
132/132 ━━━━━━━━━━━━━━━━━━━━ 203s 609ms/step - accuracy: 0.6603 - loss: 0.9501 - val_accuracy: 0.2500 - val_loss: 1.3969
Epoch 2/60
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - accuracy: 0.9585 - loss: 0.4645
Epoch 2: val_accuracy improved from 0.25000 to 0.49111, saving model to carenet.keras
132/132 ━━━━━━━━━━━━━━━━━━━━ 15s 113ms/step - accuracy: 0.9585 - loss: 0.4645 - val_accuracy: 0.4911 - val_loss: 1.2225
Epoch 3/60
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - accuracy: 0.9822 - loss: 0.4134
Epoch 3: val_accuracy improved from 0.49111 to 0.72444, saving model to carenet.keras
132/132 ━━━━━━━━━━━━━━━━━━━━ 15s 114ms/step - accuracy: 0.9822 - loss: 0.4133 - val_accuracy: 0.7244 - val_loss: 0.8840
Epoch 4/60
132/132 ━━━━━━━━━━━━━━━━━━━━ 